# CS781 Phase 5 — Legal Statute Identification (LSI)

Phase 1 legal backbones → long-document **multi-label** statute identification.
Follows §11 of `Capstone Document_edited.pdf` and the professor's starter workflow:
authenticate → configure → load official data → train/adapt → evaluate → reload/verify → ZIP.
Only your Phase 1 weights, tokenizer and supplied competition data are used.

**Train fits models; dev selects checkpoints/thresholds/hyperparameters; test produces predictions.**
Outputs save automatically under **`MyDrive/cs781_phase5/`**.
Primary: **Macro-F1**. Secondary: **Micro-F1 and Subset Accuracy**.
Predictions are a list of `{"id": original_id, "pred": [integer_statute_ids]}` objects.

Six core runs cover all four Phase 1 origins, full/partial/adapters/compulsory LoRMA and compulsory KD.
Seven optional profiles and thirteen bounded tuning trials broaden comparisons. More trials consume
more GPU time and may overfit dev; no leaderboard improvement is guaranteed.

## 1. Install dependencies

In [ ]:
%pip -q install "tokenizers>=0.20,<0.24" "huggingface_hub>=0.26,<3" "tqdm>=4.66" "numpy>=1.26" "matplotlib>=3.8"

## 2. Mount Drive, authenticate with an HF token, and check GPU

In [ ]:
import os, json, math, random, time, zipfile, hashlib, gc, copy, getpass, platform
from pathlib import Path
from contextlib import nullcontext
from collections import Counter
from bisect import bisect_left, bisect_right
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from tokenizers import ByteLevelBPETokenizer

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive')
else:
    BASE_DIR = Path.cwd()

PHASE1_DIR = BASE_DIR / 'cs781_phase1' if IN_COLAB else BASE_DIR
PROJECT_DIR = BASE_DIR / 'cs781_phase5'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(PROJECT_DIR / 'hf_cache')
from huggingface_hub import hf_hub_download, login

HF_TOKEN = os.environ.get('HF_TOKEN')
if IN_COLAB and not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get('HF')
    except Exception:
        pass
def validate_hf_token(value):
    # Never print or include the credential in an error message.
    if not isinstance(value, str):
        raise TypeError('HF token must be a plain text string, not ' + type(value).__name__ + '.')
    value = value.strip()
    if not value.startswith('hf_'):
        raise ValueError('Paste only the Hugging Face personal access token starting with hf_.')
    return value

if HF_TOKEN:
    try:
        HF_TOKEN = validate_hf_token(HF_TOKEN)
    except (TypeError, ValueError) as error:
        print(str(error))
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste only your hf_ token (hidden): ')
HF_TOKEN = validate_hf_token(HF_TOKEN)
login(token=HF_TOKEN, add_to_git_credential=False)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('CPU is suitable for checks; select a Colab GPU before full training.')
print('Phase 1 input:', PHASE1_DIR)
print('Phase 5 output:', PROJECT_DIR)

# Atomic writes avoid leaving partially written JSON/checkpoints after interruption.
def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def save_torch(data, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
    torch.save(data, temporary)
    os.replace(temporary, path)

def amp_context():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if DEVICE.type == 'cuda' else nullcontext()

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

## 3. All experiment, hyperparameter and calibration controls — edit here

First verify `decoder_full` with `RUN_ALL=False`. Then use `RUN_ALL=True` for the six core profiles.
For independent Colab runtimes choose one `EXPERIMENT`, leave all sweep flags false, and keep
`BUILD_ENSEMBLE=False`. Finish Section 5 in one runtime before others prepare shared caches.
Run `lightweight_kd` **after a larger completed Phase 5 teacher exists**. A Phase 4 teacher is not valid.
Use a new `RUN_TAG` whenever changing training settings. Matching tag/settings resume automatically.
Calibration is automatic on dev and does not retrain models within an epoch.

In [ ]:
SEED = 42
EXPERIMENT = 'decoder_full'
RUN_ALL = False
RUN_EXTRA_APPROACHES = False
RUN_TUNING = False
BUILD_ENSEMBLE = False  # Enable after two or more runs have finished.
RUN_TAG = 'v1'
RESUME = True
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
EPOCHS = 6
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
CHUNK_LENGTH = 384
CHUNK_OVERLAP = 64
MAX_CHUNKS = 4  # 0 = every overlapping window; substantially more compute.
CHUNK_BATCH_SIZE = 4
TRAIN_CHUNK_POLICY = 'spread'
EVAL_CHUNK_POLICY = 'spread'
DOC_POOLING = 'attention'  # attention, mean, max, mean_max, transformer
DOC_HIDDEN = 128
HEAD_DROPOUT = 0.15
HEAD_LR_MULTIPLIER = 5.0
USE_CLASS_WEIGHTS = True
CLASS_WEIGHT_POWER = 0.5
POS_WEIGHT_CAP = 20.0
LOSS_NAME = 'bce'  # bce, focal, asymmetric
FOCAL_GAMMA = 2.0
ASL_GAMMA_POS = 0.0
ASL_GAMMA_NEG = 4.0
ASL_CLIP = 0.05
LABEL_SMOOTHING = 0.0
INITIALIZE_LABEL_PRIORS = True
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1
EARLY_STOPPING_PATIENCE = 3
GRADIENT_CHECKPOINTING = True
ADAPTER_RANK = 8
ADAPTER_ALPHA = 16.0
BOTTLENECK_DIM = 64
PARTIAL_LAYERS = 2
UNFREEZE_LAYERS_PER_EPOCH = 2
KD_WEIGHT = 0.5
KD_TEMPERATURE = 2.0
TEACHER_RUN_DIR = None  # Optional absolute Drive folder of a completed Phase 5 teacher.
SAVE_EVERY_STEPS = 100
MAX_OPTIMIZER_STEPS = None  # Use a new tag and a small cap only for smoke checks.
NUM_WORKERS = 0
TUNE_THRESHOLD = True
THRESHOLD_MODE = 'auto'  # global, per_label, auto (dev-select global vs shrunk per-label)
THRESHOLD_GRID = [round(float(x), 4) for x in np.linspace(0.02, 0.98, 49)] + [0.5]
MIN_DEV_POSITIVES = 3
THRESHOLD_SHRINKAGE = 10.0
ALLOW_EMPTY_PREDICTIONS = True
# PDF says the evaluator derives its dimension from reference data, but supplies no evaluator code.
# reference_range = IDs 0..max(gold ID); reference_present = observed gold IDs; known = full catalog.
# All three Macro-F1 variants are reported. Change only if the official evaluator clarifies its rule.
METRIC_LABEL_SCOPE = 'reference_range'
STATUTE_IDS = None  # Optional official valid-ID list; otherwise infer from TRAIN + DEV, never test labels.
SPLIT_OVERLAP_POLICY = 'exclude_train'  # exclude_train (default), error, report
PHASE1_HEADS = {'decoder_only': 12, 'encoder_decoder': 8, 'hybrid_prefixlm': 10, 'lightweight': 12}

# Four architectures, three adaptation methods plus compulsory KD. LoRMA is compulsory.
CORE_EXPERIMENTS = {
    'decoder_full': {'architecture': 'decoder_only', 'method': 'full', 'lr': 2e-5},
    'encoder_partial': {'architecture': 'encoder_decoder', 'method': 'partial', 'lr': 5e-5},
    'hybrid_lorma': {'architecture': 'hybrid_prefixlm', 'method': 'lorma', 'lr': 2e-4},
    'hybrid_adapters': {'architecture': 'hybrid_prefixlm', 'method': 'adapters', 'lr': 2e-4},
    'lightweight_lorma': {'architecture': 'lightweight', 'method': 'lorma', 'lr': 2e-4},
    'lightweight_kd': {'architecture': 'lightweight', 'method': 'kd', 'lr': 5e-5},
}
EXTRA_EXPERIMENTS = {
    'lightweight_full': {'architecture': 'lightweight', 'method': 'full', 'lr': 2e-5},
    'encoder_full': {'architecture': 'encoder_decoder', 'method': 'full', 'lr': 2e-5},
    'decoder_lorma': {'architecture': 'decoder_only', 'method': 'lorma', 'lr': 2e-4},
    'encoder_adapters': {'architecture': 'encoder_decoder', 'method': 'adapters', 'lr': 2e-4},
    'decoder_gradual': {'architecture': 'decoder_only', 'method': 'gradual', 'lr': 2e-5},
    'decoder_linear_probe': {'architecture': 'decoder_only', 'method': 'linear_probe', 'lr': 2e-4},
    'hybrid_full': {'architecture': 'hybrid_prefixlm', 'method': 'full', 'lr': 2e-5},
}
EXPERIMENTS = {**CORE_EXPERIMENTS, **EXTRA_EXPERIMENTS}
TUNE_BASE_EXPERIMENT = 'hybrid_lorma'  # Change to your strongest measured dev profile.
TUNING_TRIALS = [
    {'name': 'lr_low', 'overrides': {'lr': 1e-4}},
    {'name': 'more_chunks', 'overrides': {'max_chunks': 8}},
    {'name': 'longer_chunks', 'overrides': {'chunk_length': 512}},
    {'name': 'mean_max', 'overrides': {'pooling': 'mean_max'}},
    {'name': 'doc_transformer', 'overrides': {'pooling': 'transformer'}},
    {'name': 'head_tail', 'overrides': {'train_chunk_policy': 'head_tail', 'eval_chunk_policy': 'head_tail'}},
    {'name': 'random_chunks', 'overrides': {'train_chunk_policy': 'random_mix'}},
    {'name': 'no_pos_weights', 'overrides': {'use_class_weights': False}},
    {'name': 'focal', 'overrides': {'loss_name': 'focal'}},
    {'name': 'asymmetric', 'overrides': {'loss_name': 'asymmetric'}},
    {'name': 'global_threshold', 'overrides': {'threshold_mode': 'global'}},
    {'name': 'label_smoothing', 'overrides': {'label_smoothing': 0.02}},
    {'name': 'label_graph', 'overrides': {'label_graph_weight': 0.2}},
]
ENSEMBLE_TOP_K = 3
ENSEMBLE_PAIR_WEIGHTS = [0.25, 0.5, 0.75]
assert EXPERIMENT in EXPERIMENTS and TUNE_BASE_EXPERIMENT in EXPERIMENTS
assert EPOCHS > 0 and BATCH_SIZE > 0 and GRAD_ACCUM_STEPS > 0 and SAVE_EVERY_STEPS > 0
assert 0 <= KD_WEIGHT <= 1 and KD_TEMPERATURE > 0
assert METRIC_LABEL_SCOPE in ('known', 'reference_present', 'reference_range')
assert MAX_OPTIMIZER_STEPS is None or MAX_OPTIMIZER_STEPS > 0
seed_all(SEED)

## 4. Download all official splits, validate formats and build the statute catalog

Supports JSON/JSONL, explicit-column CSV and safe literal exports despite `.json` filenames.
One forced fresh download follows a parsing failure. `dataset_loading_report.json` names its split.
Train/dev repeated IDs retain all instances; test IDs/types remain unchanged and must be unique.
Exact nonempty training copies of dev/test documents are excluded under the default
`SPLIT_OVERLAP_POLICY='exclude_train'`. Every dev/test case remains. Counts and ID/hash provenance
save to `split_audit.json` and `excluded_train_instances.json`; weights/priors are recomputed afterward.
The label catalog comes from train+dev (or `STATUTE_IDS`), never private test labels. IDs can have gaps.
Labels absent from train are reported rather than assigned invented training targets.

In [ ]:
FILES = {'train':'Phase_5_LSI/Train/train.json','dev':'Phase_5_LSI/Train/dev.json',
         'test':'Phase_5_LSI/Test/test.json'}
local_files={}
for split,filename in tqdm(FILES.items(),desc='Downloading LSI splits',unit='file'):
    local_files[split]=Path(hf_hub_download(repo_id=HF_DATASET_REPO,filename=filename,
                                         repo_type='dataset',token=HF_TOKEN))

def parse_case_file(path):
    import ast, csv, io
    path=Path(path)
    with path.open('rb') as stream:
        signature=stream.read(4)
    encoding='utf-16' if signature.startswith((b'\xff\xfe',b'\xfe\xff')) else 'utf-8-sig'
    try:
        content=path.read_text(encoding=encoding)
    except UnicodeError as exc:
        raise ValueError('File is not UTF-8/UTF-16 text; possibly an incomplete or non-text download.') from exc
    stripped=content.strip()
    if not stripped:
        raise ValueError('Downloaded file is empty or whitespace-only.')
    if stripped.startswith('version https://git-lfs.github.com/spec/v1'):
        raise ValueError('Downloaded file is a Git LFS pointer, not the dataset contents.')
    if stripped.lower().startswith(('<!doctype html','<html','<?xml')):
        raise ValueError('Downloaded file is an HTML/XML response, not case data.')
    try:
        return json.loads(content), 'json'
    except json.JSONDecodeError as exc:
        json_error=f'JSON parse failed at line {exc.lineno}, column {exc.colno}.'
    # Newline-delimited objects are accepted only if every nonblank line is valid.
    lines=io.StringIO(content)
    first=next((line.strip() for line in lines if line.strip()),'')
    if first.startswith('{'):
        try:
            initial=json.loads(first)
            if isinstance(initial,dict):
                rows=[initial]
                for number,line in enumerate(lines,2):
                    if not line.strip(): continue
                    row=json.loads(line)
                    if not isinstance(row,dict): raise ValueError('JSON Lines must contain objects.')
                    rows.append(row)
                return rows,'jsonl'
        except (json.JSONDecodeError,ValueError):
            pass
    # Some exports are CSV despite their .json suffix. Require known columns.
    csv_reader=csv.reader(io.StringIO(content))
    try:
        header=next(csv_reader)
    except (StopIteration,csv.Error):
        header=[]
    if 'id' in header and 'text' in header:
        csv.field_size_limit(min(__import__('sys').maxsize,2**31-1))
        reader=csv.DictReader(io.StringIO(content),strict=True)
        rows=[]
        try:
            for row in reader:
                if None in row or any(value is None for value in row.values()):
                    raise ValueError('CSV row has missing or extra columns.')
                rows.append(row)
        except csv.Error as exc:
            raise ValueError('Malformed CSV case export.') from exc
        return rows,'csv'
    # Safe parsing only; never execute downloaded Python code.
    if stripped.lstrip('[').lstrip().startswith(('{'+chr(39),'{"')):
        try:
            literal=ast.literal_eval(stripped)
            if isinstance(literal,(list,dict)):
                return literal,'python_literal'
        except (SyntaxError,ValueError,TypeError,RecursionError):
            pass
    raise ValueError(json_error+' Unrecognized or malformed case-file format; no records were skipped.')

loading_report={}
def load_case_split(split):
    attempts=[]
    for attempt in range(2):
        path=local_files[split]
        detail={'attempt':attempt+1,'bytes':path.stat().st_size}
        print(f'Parsing {split}: {FILES[split]} ({detail["bytes"]:,} bytes)',flush=True)
        try:
            raw,format_name=parse_case_file(path)
        except ValueError as exc:
            detail.update(status='parse_failed',reason=str(exc)); attempts.append(detail)
            loading_report[split]=attempts
            save_json(loading_report,PROJECT_DIR/'dataset_loading_report.json')
            if attempt==0:
                print(f'{split}: {exc} Retrying with a fresh Hugging Face download.',flush=True)
                local_files[split]=Path(hf_hub_download(repo_id=HF_DATASET_REPO,filename=FILES[split],
                    repo_type='dataset',token=HF_TOKEN,force_download=True))
                continue
            raise ValueError(f'{split} ({FILES[split]}) could not be parsed after a fresh download: {exc} '
                             'See dataset_loading_report.json in the Phase 4 Drive folder.') from exc
        detail.update(status='parsed',format=format_name); attempts.append(detail)
        loading_report[split]=attempts
        save_json(loading_report,PROJECT_DIR/'dataset_loading_report.json')
        print(f'{split}: detected {format_name}',flush=True)
        return normalize_cases(raw,split!='test',split=split)


def case_key(case_id):
    if isinstance(case_id, bool) or not isinstance(case_id, (str, int)):
        raise ValueError('Case IDs must be strings or integers.')
    return str(case_id)


def parse_statutes(value):
    import ast
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            try:
                value = ast.literal_eval(value)
            except (SyntaxError, ValueError) as exc:
                raise ValueError('labels must contain a list of integer statute IDs.') from exc
    if not isinstance(value, list):
        raise ValueError('Expected a multi-label list, not a single class.')
    if any(type(x) is not int or x < 0 for x in value):
        raise ValueError('Statute IDs must be nonnegative integers (booleans are invalid).')
    return sorted(set(value))


def normalize_cases(raw, labeled, split=None):
    if isinstance(raw, dict) and 'id' in raw and 'text' in raw:
        entries = [(None, raw)]
    elif isinstance(raw, dict) and isinstance(raw.get('data'), list):
        entries = [(None, row) for row in raw['data']]
    elif isinstance(raw, dict):
        entries = list(raw.items())
    elif isinstance(raw, list):
        entries = [(None, row) for row in raw]
    else:
        raise ValueError('Expected case objects or an ID-keyed mapping.')
    result = {}
    for index, (outer_id, record) in enumerate(entries):
        if isinstance(record, str) and not labeled and outer_id is not None:
            record = {'id': outer_id, 'text': record}
        if not isinstance(record, dict):
            raise ValueError(f'{split}: invalid case object at row {index}.')
        original_id = record.get('id', outer_id)
        key = case_key(original_id)
        if outer_id is not None and str(outer_id) != key:
            raise ValueError('Mapping key disagrees with embedded case ID.')
        text = record.get('text')
        if isinstance(text, list) and all(isinstance(x, str) for x in text):
            text = '\n\n'.join(text)
        if not isinstance(text, str):
            raise ValueError(f'{split} row {index}: text must be a string or ordered string fragments.')
        labels = parse_statutes(record.get('labels', record.get('label'))) if labeled else None
        internal_key = f'{split}:row:{index}' if split in ('train', 'dev') else key
        if internal_key in result:
            raise ValueError(f'{split}: duplicate original test ID {key}; no rows were overwritten.')
        result[internal_key] = {'id': original_id, 'text': text, 'label': labels}
    return result


def resolve_split_overlaps(data, policy):
    if policy not in ('exclude_train', 'error', 'report'):
        raise ValueError('SPLIT_OVERLAP_POLICY must be exclude_train, error or report.')
    indexes = {}
    for split, docs in data.items():
        index = {}
        for key, doc in tqdm(docs.items(), desc='Auditing exact texts: ' + split, unit='case'):
            if doc['text'].strip():
                digest = hashlib.sha256(doc['text'].encode()).hexdigest()
                index.setdefault(digest, []).append(key)
        indexes[split] = index
    report = {'policy': policy, 'source_counts': {split: len(docs) for split, docs in data.items()}}
    for a, b in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:
        shared_ids = {case_key(d['id']) for d in data[a].values()} & {case_key(d['id']) for d in data[b].values()}
        shared_texts = set(indexes[a]) & set(indexes[b])
        examples = []
        for digest in sorted(shared_texts)[:20]:
            akeys = indexes[a][digest]; bkeys = indexes[b][digest]
            example = {'sha256': digest, 'characters': len(data[a][akeys[0]]['text']),
                       a + '_ids': [data[a][key]['id'] for key in akeys[:5]],
                       b + '_ids': [data[b][key]['id'] for key in bkeys[:5]]}
            if a != 'test' and b != 'test':
                label_sets = {tuple(data[split][key]['label']) for split, keys in [(a, akeys), (b, bkeys)] for key in keys}
                example['different_label_sets'] = len(label_sets) > 1
            examples.append(example)
        report[a + '/' + b] = {'shared_original_ids': len(shared_ids), 'identical_texts': len(shared_texts),
                              a + '_matching_instances': sum(len(indexes[a][h]) for h in shared_texts),
                              b + '_matching_instances': sum(len(indexes[b][h]) for h in shared_texts),
                              'examples': examples}
        print(f"Split audit {a}/{b}: {len(shared_ids)} shared ID values; {len(shared_texts)} identical nonempty texts.")
    heldout = set(indexes['dev']) | set(indexes['test'])
    remove = {key for digest in set(indexes['train']) & heldout for key in indexes['train'][digest]}
    filtered = dict(data)
    removed = []
    if policy == 'exclude_train':
        removed = [{'instance_key': key, 'id': doc['id'], 'sha256': hashlib.sha256(doc['text'].encode()).hexdigest()}
                   for key, doc in data['train'].items() if key in remove]
        filtered['train'] = {key: doc for key, doc in data['train'].items() if key not in remove}
    report['excluded_train_instances'] = len(removed)
    report['excluded_train_examples'] = removed[:20]
    report['retained_counts'] = {split: len(docs) for split, docs in filtered.items()}
    report['remaining_train_instances_matching_heldout_text'] = 0 if policy == 'exclude_train' else len(remove)
    save_json(report, PROJECT_DIR / 'split_audit.json')
    save_json(removed, PROJECT_DIR / 'excluded_train_instances.json')
    if policy == 'error' and any(report[a + '/' + b]['identical_texts'] for a, b in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]):
        raise ValueError('Identical nonempty texts across splits; see split_audit.json. Use exclude_train to remove held-out copies from training.')
    if not filtered['train']:
        raise ValueError('No training cases remain after overlap filtering; inspect split_audit.json before choosing another policy.')
    if policy == 'exclude_train':
        retained_hashes = {hashlib.sha256(doc['text'].encode()).hexdigest() for doc in filtered['train'].values() if doc['text'].strip()}
        assert not (retained_hashes & heldout), 'Overlap filtering left held-out copies in training.'
        print(f"Excluded {len(removed)} training instances matching dev/test text; kept {len(filtered['train'])} training instances.")
    if report['dev/test']['identical_texts']:
        print('Official dev/test contain identical texts. All dev/test cases and IDs are retained; overlap counts are recorded in split_audit.json.')
    if policy == 'report' and remove:
        print('Report-only policy: matching training instances are retained. Dev scores include exact-text overlap with training.')
    return filtered, report


DATA = {split: load_case_split(split) for split in FILES}
# Catalog IDs come from supplied labeled files, including excluded training instances.
observed = {x for split in ('train', 'dev') for row in DATA[split].values() for x in row['label']}
if not observed:
    raise ValueError('No statute labels in TRAIN/DEV.')
if STATUTE_IDS is not None:
    LABEL_IDS = parse_statutes(STATUTE_IDS)
    if not observed <= set(LABEL_IDS):
        raise ValueError('Observed labels are absent from the supplied official statute catalog.')
else:
    LABEL_IDS = sorted(observed)
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABEL_IDS)}
NUM_LABELS = len(LABEL_IDS)
DATA, split_audit = resolve_split_overlaps(DATA, SPLIT_OVERLAP_POLICY)


def encode_labels(labels):
    array = np.zeros(NUM_LABELS, dtype=np.float32)
    if labels is not None:
        for label in labels:
            array[LABEL_TO_INDEX[label]] = 1.0
    return array


TRAIN_COUNTS = np.zeros(NUM_LABELS, dtype=np.int64)
DEV_COUNTS = np.zeros(NUM_LABELS, dtype=np.int64)
audit = {}
for split, docs in DATA.items():
    if not docs:
        raise ValueError(f'Empty {split} split.')
    counter = Counter(case_key(row['id']) for row in docs.values())
    audit[split] = {'instances': len(docs), 'unique_ids': len(counter),
                    'repeated_ids': sum(v > 1 for v in counter.values()),
                    'empty_texts': sum(not row['text'].strip() for row in docs.values())}
    if split != 'test':
        counts = TRAIN_COUNTS if split == 'train' else DEV_COUNTS
        for row in docs.values():
            counts += encode_labels(row['label']).astype(np.int64)
        audit[split]['mean_labels'] = float(np.mean([len(row['label']) for row in docs.values()]))
    print(split, audit[split])
fingerprint = hashlib.sha256(json.dumps([LABEL_IDS, SPLIT_OVERLAP_POLICY]).encode())
for split in sorted(DATA):
    for key, doc in DATA[split].items():
        fingerprint.update(json.dumps([split, key, doc], sort_keys=True, ensure_ascii=False).encode())
DATA_FINGERPRINT = fingerprint.hexdigest()
save_json({'files': FILES, 'fingerprint': DATA_FINGERPRINT, 'counts': audit,
           'source_counts': split_audit['source_counts'], 'overlap_policy': SPLIT_OVERLAP_POLICY,
           'excluded_train_instances': split_audit['excluded_train_instances']}, PROJECT_DIR / 'dataset_manifest.json')
save_json({'label_ids': LABEL_IDS, 'num_labels': NUM_LABELS, 'source': 'official override' if STATUTE_IDS is not None else 'train+dev',
           'train_counts': TRAIN_COUNTS.tolist(), 'dev_counts': DEV_COUNTS.tolist(),
           'unseen_in_train': [x for x, n in zip(LABEL_IDS, TRAIN_COUNTS) if not n],
           'absent_from_dev': [x for x, n in zip(LABEL_IDS, DEV_COUNTS) if not n]}, PROJECT_DIR / 'label_catalog.json')
print('Output labels:', NUM_LABELS, '; IDs:', LABEL_IDS)
print('Labels absent from TRAIN:', [x for x, n in zip(LABEL_IDS, TRAIN_COUNTS) if not n])

## 5. Shared Phase 1 BPE, Drive token caches and overlapping document windows

No external tokenizer is downloaded. Default: four spread windows including the beginning/end,
not full document coverage. `chunk_coverage.json` reports actual coverage. Compare `MAX_CHUNKS=8`
or `0` (all windows), head/tail selection, random training windows and hierarchical pooling.
Each case still receives exactly one multi-label prediction. Empty texts receive an UNK chunk.

In [ ]:
TOKENIZER_DIR=PHASE1_DIR/'bpe_tokenizer'
for filename in ['vocab.json','merges.txt']:
    if not (TOKENIZER_DIR/filename).is_file():
        raise FileNotFoundError(TOKENIZER_DIR/filename)
tok=ByteLevelBPETokenizer(str(TOKENIZER_DIR/'vocab.json'),str(TOKENIZER_DIR/'merges.txt'))
PAD_ID,BOS_ID,EOS_ID,UNK_ID,SEP_ID=[tok.token_to_id(s) for s in ['<pad>','<bos>','<eos>','<unk>','<sep>']]
assert all(v is not None for v in [PAD_ID,BOS_ID,EOS_ID,UNK_ID,SEP_ID])
VOCAB_SIZE_ACTUAL=tok.get_vocab_size()
TOKENIZER_HASH=hashlib.sha256(b''.join((TOKENIZER_DIR/f).read_bytes() for f in ['vocab.json','merges.txt'])).hexdigest()
import shutil
(PROJECT_DIR/'bpe_tokenizer').mkdir(exist_ok=True)
for filename in ['vocab.json','merges.txt']:
    shutil.copy2(TOKENIZER_DIR/filename,PROJECT_DIR/'bpe_tokenizer'/filename)
class TokenCache:
    def __init__(self,binary_path,index):
        self.index=index
        expected=sum(size for start,size in index.values())
        if binary_path.stat().st_size!=4*expected:
            raise ValueError('Truncated token-cache file: remove/rebuild this cache before training.')
        self.tokens=np.memmap(binary_path,dtype='<i4',mode='r') if expected else np.empty(0,dtype=np.int32)
    def __getitem__(self,key):
        start,size=self.index[key]
        return self.tokens[start:start+size]
    def __len__(self):
        return len(self.index)

TOKENS={}
for split,docs in DATA.items():
    stem=f'tokens_{split}_{DATA_FINGERPRINT[:12]}_{TOKENIZER_HASH[:12]}'
    binary_path=PROJECT_DIR/(stem+'.bin'); index_path=PROJECT_DIR/(stem+'.json')
    if binary_path.is_file() and index_path.is_file():
        index=json.loads(index_path.read_text())
        if set(index)!=set(docs): raise ValueError('Token cache IDs disagree with current split.')
        print('Reused disk-backed token cache:',split)
    else:
        index={}; offset=0; temporary=binary_path.with_name(binary_path.name+'.'+__import__('uuid').uuid4().hex+'.tmp')
        with open(temporary,'wb',buffering=8*1024*1024) as stream:
            for key,doc in tqdm(docs.items(),desc='Tokenizing '+split,unit='case'):
                ids=np.asarray(tok.encode(doc['text'],add_special_tokens=False).ids,dtype='<i4')
                stream.write(ids.tobytes()); index[key]=[offset,len(ids)]; offset+=len(ids)
        os.replace(temporary,binary_path); save_json(index,index_path)
    # Stage once onto Colab's local disk; canonical token caches and all results persist in Drive.
    active_path=binary_path
    if IN_COLAB:
        staging=Path('/content/cs781_phase5_token_cache'); staging.mkdir(exist_ok=True)
        active_path=staging/binary_path.name
        if not active_path.is_file() or active_path.stat().st_size!=binary_path.stat().st_size:
            temporary=active_path.with_name(active_path.name+'.'+__import__('uuid').uuid4().hex+'.tmp')
            with open(binary_path,'rb') as source_stream,open(temporary,'wb') as target_stream:
                with tqdm.wrapattr(source_stream,'read',total=binary_path.stat().st_size,desc='Staging '+split,bytes=True) as monitored:
                    shutil.copyfileobj(monitored,target_stream,length=8*1024*1024)
            os.replace(temporary,active_path)
    TOKENS[split]=TokenCache(active_path,index)

def window_ranges(n,length,overlap):
    size=length-2
    if n==0:
        return [(0,0)]
    ranges=[]; start=0
    while True:
        end=min(start+size,n); ranges.append((start,end))
        if end==n:
            return ranges
        start+=size-overlap

def select_ranges(ranges,limit,policy,key='',epoch=0,seed=SEED):
    if policy=='all' or limit==0 or len(ranges)<=limit:
        return list(ranges)
    if policy=='head':
        return ranges[:limit]
    if policy=='tail':
        return ranges[-limit:]
    if policy=='head_tail':
        first=(limit+1)//2
        return ranges[:first]+(ranges[-(limit-first):] if limit-first else [])
    if policy=='random_mix':
        stable=int(hashlib.sha256(key.encode()).hexdigest()[:12],16)
        rng=random.Random(seed+epoch*1000003+stable)
        if limit==1:
            return [ranges[rng.randrange(len(ranges))]]
        chosen=[0,len(ranges)-1]+rng.sample(list(range(1,len(ranges)-1)),limit-2)
        return [ranges[i] for i in sorted(chosen)]
    if policy=='spread':
        indices=np.linspace(0,len(ranges)-1,limit).round().astype(int).tolist()
        return [ranges[i] for i in indices]
    raise ValueError('Unknown chunk policy '+policy)

def covered_tokens(ranges):
    total=0; previous=0
    for start,end in sorted(ranges):
        total+=max(0,end-max(start,previous)); previous=max(previous,end)
    return total

class CaseDataset(Dataset):
    def __init__(self,split,cfg,training=False):
        self.split=split; self.cfg=cfg; self.training=training; self.epoch=0
        self.keys=list(DATA[split])
    def __len__(self):
        return len(self.keys)
    def __getitem__(self,index):
        key=self.keys[index]; tokens=TOKENS[self.split][key]
        ranges=window_ranges(len(tokens),self.cfg['chunk_length'],self.cfg['chunk_overlap'])
        policy=self.cfg['train_chunk_policy'] if self.training else self.cfg['eval_chunk_policy']
        selected=select_ranges(ranges,self.cfg['max_chunks'],policy,key,self.epoch,self.cfg['seed'])
        chunks=[[BOS_ID]+(tokens[s:e].tolist() or [UNK_ID])+[EOS_ID] for s,e in selected]
        positions=[[s/max(1,len(tokens)),e/max(1,len(tokens))] for s,e in selected]
        coverage=covered_tokens(selected)/max(1,len(tokens)) if len(tokens) else 1.0
        stats=[math.log1p(len(tokens))/10,math.log1p(len(ranges))/5,coverage,len(selected)/max(1,len(ranges))]
        return {'key':key,'chunks':chunks,'positions':positions,'stats':stats,
                'label':DATA[self.split][key]['label'],'coverage':coverage}

def collate_cases(rows):
    chunks=[(owner,ids,pos) for owner,row in enumerate(rows) for ids,pos in zip(row['chunks'],row['positions'])]
    length=max(len(ids) for _,ids,_ in chunks)
    ids=torch.full((len(chunks),length),PAD_ID,dtype=torch.long)
    mask=torch.zeros_like(ids,dtype=torch.bool)
    content=torch.zeros_like(mask)
    for i,(_,values,_) in enumerate(chunks):
        ids[i,:len(values)]=torch.tensor(values); mask[i,:len(values)]=True
        content[i,1:len(values)-1]=True
    return {'input_ids':ids,'attention_mask':mask,'target_mask':content,
            'chunk_positions':torch.tensor([p for _,_,p in chunks],dtype=torch.float32),
            'owners':torch.tensor([o for o,_,_ in chunks]),
            'stats':torch.tensor([r['stats'] for r in rows],dtype=torch.float32),
            'labels':torch.tensor(np.stack([encode_labels(r['label']) for r in rows]),dtype=torch.float32),
            'keys':[r['key'] for r in rows]}

## 6. Strictly checkpoint-compatible Phase 1 architectures — no pretraining rerun

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def apply_rope(q, k, cos, sin):
    q_ = (q * cos) + (rotate_half(q) * sin)
    k_ = (k * cos) + (rotate_half(k) * sin)
    return q_, k_


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, max_len):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)).bool(), persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class DecoderOnlyGPT(nn.Module):
    """Architecture 1: classic decoder-only Transformer LM."""

    def __init__(self, vocab_size, n_layer, n_embd, n_head, d_ff, max_len=2048):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([GPTBlock(n_embd, n_head, d_ff, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached generation (recomputes full forward pass each step).
        # Positions are re-based to 0 each call since context is re-clipped to max_len.
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
class RelativePositionBias(nn.Module):
    def __init__(self, n_head, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, n_head)

    @staticmethod
    def _bucket(rel_pos, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -rel_pos
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6) / math.log(max_distance / max_exact) * (num_buckets - max_exact)
        ).long()
        val_large = torch.clamp(val_large, max=num_buckets - 1)
        ret = ret + torch.where(is_small, n, val_large)
        return ret

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        buckets = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(buckets)                    # (q_len, k_len, n_head)
        return values.permute(2, 0, 1).unsqueeze(0)          # (1, n_head, q_len, k_len)


class RelPosAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_q, x_kv, bias=None, causal=False, key_padding_mask=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, Tq, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if bias is not None:
            att = att + bias
        if causal:
            causal_mask = torch.tril(torch.ones(Tq, Tk, device=x_q.device)).bool()
            att = att.masked_fill(~causal_mask, float("-inf"))
        if key_padding_mask is not None:
            pad_keep = key_padding_mask[:, None, None, :].bool()   # (B,1,1,Tk), 1 = keep
            att = att.masked_fill(~pad_keep, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.o_proj(out)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, bias, key_padding_mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, bias=bias, causal=False, key_padding_mask=key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = RelPosAttention(d_model, n_head)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, enc_out, self_bias, enc_padding_mask):
        h = self.ln1(x)
        x = x + self.self_attn(h, h, bias=self_bias, causal=True)
        x = x + self.cross_attn(self.ln2(x), enc_out, bias=None, causal=False, key_padding_mask=enc_padding_mask)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderTransformer(nn.Module):
    """Architecture 2: T5-style encoder-decoder Transformer with relative position bias."""

    def __init__(self, vocab_size, n_enc_layer, n_dec_layer, d_model, n_head, d_ff):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_head, d_ff) for _ in range(n_enc_layer)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_head, d_ff) for _ in range(n_dec_layer)])
        self.enc_rel_bias = RelativePositionBias(n_head, bidirectional=True)
        self.dec_rel_bias = RelativePositionBias(n_head, bidirectional=False)
        self.ln_enc_f = nn.LayerNorm(d_model)
        self.ln_dec_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, src_ids, src_mask):
        x = self.tok_emb(src_ids)
        bias = self.enc_rel_bias(src_ids.size(1), src_ids.size(1), src_ids.device)
        for layer in self.enc_layers:
            x = layer(x, bias, src_mask)
        return self.ln_enc_f(x)

    def forward(self, src_ids, src_mask, tgt_in, tgt_out=None):
        enc_out = self.encode(src_ids, src_mask)
        x = self.tok_emb(tgt_in)
        bias = self.dec_rel_bias(tgt_in.size(1), tgt_in.size(1), tgt_in.device)
        for layer in self.dec_layers:
            x = layer(x, enc_out, bias, src_mask)
        x = self.ln_dec_f(x)
        logits = self.head(x)
        loss = None
        if tgt_out is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1), ignore_index=PAD_ID)
        return loss, logits

    @torch.no_grad()
    def generate(self, src_ids, src_mask, max_new_tokens, bos_id, eos_id, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached autoregressive decoding.
        self.eval()
        enc_out = self.encode(src_ids, src_mask)
        B = src_ids.size(0)
        tgt = torch.full((B, 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self.tok_emb(tgt)
            bias = self.dec_rel_bias(tgt.size(1), tgt.size(1), tgt.device)
            h = x
            for layer in self.dec_layers:
                h = layer(h, enc_out, bias, src_mask)
            h = self.ln_dec_f(h)
            logits = self.head(h)[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            tgt = torch.cat([tgt, next_id], dim=1)
        return tgt

In [ ]:
class RoPEAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, attn_mask, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~attn_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class HybridBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = RoPEAttention(d_model, n_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, attn_mask, cos, sin):
        x = x + self.attn(self.norm1(x), attn_mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


def build_prefix_lm_mask(prompt_len, total_len, device):
    """prompt<->prompt: bidirectional. continuation->prompt: visible.
    continuation<->continuation: causal."""
    mask = torch.zeros(total_len, total_len, dtype=torch.bool, device=device)
    mask[:prompt_len, :prompt_len] = True
    mask[prompt_len:, :prompt_len] = True
    cont_len = total_len - prompt_len
    if cont_len > 0:
        mask[prompt_len:, prompt_len:] = torch.tril(torch.ones(cont_len, cont_len, device=device)).bool()
    return mask


class HybridPrefixLM(nn.Module):
    """Architecture 3: single-stack encoder/hybrid LM (RoPE, RMSNorm, SwiGLU,
    UniLM-style prefix-bidirectional + causal-continuation attention mask)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, d_ff, max_len=1536):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([HybridBlock(d_model, n_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, prompt_len, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = build_prefix_lm_mask(prompt_len, T, input_ids.device).unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, prompt_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        input_ids = prompt_ids
        prompt_len = prompt_ids.size(1)
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            cur_prompt_len = min(prompt_len, ids_cond.size(1))
            _, logits = self(ids_cond, cur_prompt_len)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
class GQAAttention(nn.Module):
    """Grouped-Query Attention: n_kv_head < n_head, reducing parameters and
    (at inference, with KV caching) memory -- the key efficiency trick here."""

    def __init__(self, d_model, n_head, n_kv_head):
        super().__init__()
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, n_head * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_head * self.d_head, d_model, bias=False)

    def forward(self, x, causal_mask, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~causal_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, mask, cos, sin):
        x = x + self.attn(self.norm1(x), mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLlama(nn.Module):
    """Architecture 4: efficient/lightweight decoder-only Transformer
    (RoPE, RMSNorm, SwiGLU, Grouped-Query Attention)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, n_kv_head, d_ff, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([LightweightBlock(d_model, n_head, n_kv_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = torch.tril(torch.ones(T, T, device=input_ids.device)).bool().unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

## 7. Hierarchical multi-label head and adaptation methods

Encoder–decoder experiments use its encoder as the document backbone. Other backbones retain their
Phase 1 attention behavior. All create a new trainable statute head with independent sigmoid outputs.
KD keeps alternating Phase 1 lightweight layers, giving a smaller Phase 1-initialized student.
LoRMA+ implements `(I + alpha/r BA)W`; adapters use one post-block residual bottleneck.

In [ ]:
def torch_load(path):
    return torch.load(path, map_location='cpu', weights_only=True)

def infer_phase1_config(architecture, state):
    vocab, dim = state['tok_emb.weight'].shape
    if vocab != VOCAB_SIZE_ACTUAL:
        raise ValueError(f'Phase 1 vocabulary {vocab} != tokenizer {VOCAB_SIZE_ACTUAL}')
    config = {'vocab_size': vocab}
    def count_layers(prefix):
        return len({int(k.split('.')[1]) for k in state if k.startswith(prefix + '.')})
    heads = PHASE1_HEADS[architecture]
    if architecture == 'decoder_only':
        config.update(n_layer=count_layers('blocks'), n_embd=dim, n_head=heads,
                      d_ff=state['blocks.0.mlp.0.weight'].shape[0], max_len=state['pos_emb.weight'].shape[0])
    elif architecture == 'encoder_decoder':
        stored_heads = state['enc_rel_bias.embedding.weight'].shape[1]
        if heads != stored_heads:
            raise ValueError('Encoder head count differs from stored relative-position bias.')
        config.update(n_enc_layer=count_layers('enc_layers'), n_dec_layer=count_layers('dec_layers'),
                      d_model=dim, n_head=heads, d_ff=state['enc_layers.0.ff.0.weight'].shape[0])
    elif architecture == 'hybrid_prefixlm':
        config.update(n_layer=count_layers('blocks'), d_model=dim, n_head=heads,
                      d_ff=state['blocks.0.ff.w1.weight'].shape[0], max_len=1152)
    else:
        head_dim = dim // heads
        config.update(n_layer=count_layers('blocks'), d_model=dim, n_head=heads,
                      n_kv_head=state['blocks.0.attn.k_proj.weight'].shape[0] // head_dim,
                      d_ff=state['blocks.0.ff.w1.weight'].shape[0], max_len=1024)
    if dim % heads:
        raise ValueError('Invalid Phase 1 attention head count.')
    return config

MODEL_CLASSES = {'decoder_only': DecoderOnlyGPT, 'encoder_decoder': EncoderDecoderTransformer,
                 'hybrid_prefixlm': HybridPrefixLM, 'lightweight': LightweightLlama}

def load_phase1(architecture):
    path = PHASE1_DIR / f'run_{architecture}' / 'final_model.pt'
    if not path.is_file():
        raise FileNotFoundError(f'Missing Phase 1 weights: {path}')
    print('Loading:', path)
    state = torch_load(path)
    config = infer_phase1_config(architecture, state)
    print('Inferred config:', config)
    backbone = MODEL_CLASSES[architecture](**config)
    backbone.load_state_dict(state, strict=True)
    del state
    gc.collect()
    original_parameters = sum(p.numel() for p in backbone.parameters())
    del backbone.head
    if architecture == 'encoder_decoder':
        del backbone.dec_layers, backbone.dec_rel_bias, backbone.ln_dec_f
    return backbone, config, original_parameters

In [ ]:
class CaseFeatures(nn.Module):
    def __init__(self, backbone, architecture):
        super().__init__()
        self.backbone = backbone
        self.architecture = architecture
        self.use_checkpointing = GRADIENT_CHECKPOINTING

    def layer_call(self, layer, *args):
        if self.training and self.use_checkpointing and any(p.requires_grad for p in layer.parameters()):
            return checkpoint(layer, *args, use_reentrant=False)
        return layer(*args)

    def forward(self, input_ids, attention_mask, target_mask):
        b = self.backbone
        length = input_ids.size(1)
        x = b.tok_emb(input_ids)
        if self.architecture == 'decoder_only':
            if length > b.max_len:
                raise ValueError('MAX_LENGTH exceeds Phase 1 positional embedding length.')
            x = x + b.pos_emb(torch.arange(length, device=input_ids.device)[None, :])
            for layer in b.blocks:
                x = self.layer_call(layer, x)
            x = b.ln_f(x)
        elif self.architecture == 'encoder_decoder':
            bias = b.enc_rel_bias(length, length, input_ids.device)
            for layer in b.enc_layers:
                x = self.layer_call(layer, x, bias, attention_mask)
            x = b.ln_enc_f(x)
        else:
            cos, sin = build_rope_cache(length, b.d_model // b.n_head, input_ids.device)
            if self.architecture == 'hybrid_prefixlm':
                # All text is a bidirectional prefix; mask PAD keys explicitly.
                mask = attention_mask[:, None, None, :].expand(-1, 1, length, -1)
            else:
                causal = torch.ones(length, length, dtype=torch.bool, device=input_ids.device).tril()
                mask = causal[None, None, :, :] & attention_mask[:, None, None, :]
            for layer in b.blocks:
                x = self.layer_call(layer, x, mask, cos, sin)
            x = b.norm_f(x)
        target_weights = (target_mask & attention_mask).to(x.dtype)
        pooled = (x * target_weights[:, :, None]).sum(1) / target_weights.sum(1, keepdim=True).clamp_min(1)
        final_index = attention_mask.sum(1)-1
        final = x[torch.arange(x.size(0), device=x.device), final_index]
        return torch.cat([pooled, final], dim=-1)

In [ ]:
class LowRankMultiplicative(nn.Module):
    # LoRMA+, Eq. 8: (I + alpha/r B A) W x + bias.
    def __init__(self,base,rank,alpha):
        super().__init__(); self.base=base; base.requires_grad_(False)
        self.A=nn.Linear(base.out_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        self.scale=alpha/rank
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
    def forward(self,x):
        wx=F.linear(x,self.base.weight,None)
        y=wx+self.scale*self.B(self.A(wx))
        return y if self.base.bias is None else y+self.base.bias

class ResidualAdapter(nn.Module):
    # One post-block residual bottleneck adapter; not an exact two-site Houlsby reproduction.
    def __init__(self,base,dim,bottleneck):
        super().__init__(); self.base=base; base.requires_grad_(False)
        self.norm=nn.LayerNorm(dim); self.down=nn.Linear(dim,bottleneck); self.up=nn.Linear(bottleneck,dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,*args):
        x=self.base(*args)
        return x+self.up(F.gelu(self.down(self.norm(x))))

class LegalCase(nn.Module):
    def __init__(self,features,cfg):
        super().__init__(); self.features=features; self.cfg=cfg
        dim=features.backbone.tok_emb.embedding_dim; hidden=cfg['doc_hidden']
        self.chunk_project=nn.Sequential(nn.LayerNorm(2*dim),nn.Linear(2*dim,hidden),nn.GELU())
        self.position_project=nn.Linear(2,hidden)
        self.gate=nn.Sequential(nn.Linear(hidden,hidden),nn.Tanh(),nn.Linear(hidden,1))
        self.document_encoder=None
        if cfg['pooling']=='transformer':
            layer=nn.TransformerEncoderLayer(hidden,4,4*hidden,dropout=cfg['dropout'],batch_first=True,norm_first=True)
            self.document_encoder=nn.TransformerEncoder(layer,1,enable_nested_tensor=False)
        n_features=4 if cfg['pooling']=='mean_max' else 3
        self.classifier=nn.Sequential(nn.LayerNorm(n_features*hidden+4),nn.Linear(n_features*hidden+4,hidden),
            nn.GELU(),nn.Dropout(cfg['dropout']),nn.Linear(hidden,cfg['num_labels']))
        train_y=torch.tensor(gold_matrix('train'),dtype=torch.float32)
        co=train_y.T@train_y
        co.fill_diagonal_(0)
        graph=co/co.sum(0,keepdim=True).clamp_min(1)
        self.register_buffer('label_graph',graph if cfg['label_graph_weight'] else torch.zeros_like(graph))
    def forward(self,input_ids,attention_mask,target_mask,chunk_positions,owners,stats):
        chunks=[]
        for first in range(0,len(input_ids),self.cfg['chunk_batch_size']):
            stop=first+self.cfg['chunk_batch_size']
            features=self.features(input_ids[first:stop],attention_mask[first:stop],target_mask[first:stop])
            chunks.append(self.chunk_project(features)+self.position_project(chunk_positions[first:stop]))
        x=torch.cat(chunks,dim=0)
        documents=[]
        for owner in range(len(stats)):
            h=x[owners==owner]
            if not len(h):
                raise ValueError('Missing chunks for a case.')
            pooling=self.cfg['pooling']
            if pooling=='attention':
                pooled=(h*torch.softmax(self.gate(h).squeeze(-1),dim=0)[:,None]).sum(0)
            elif pooling=='mean': pooled=h.mean(0)
            elif pooling=='max': pooled=h.max(0).values
            elif pooling=='mean_max': pooled=torch.cat([h.mean(0),h.max(0).values])
            elif pooling=='transformer': pooled=self.document_encoder(h[None])[0].mean(0)
            else: raise ValueError('Unknown document pooling.')
            documents.append(torch.cat([pooled,h[0],h[-1],stats[owner]]))
        logits=self.classifier(torch.stack(documents))
        return logits+self.cfg['label_graph_weight']*(torch.sigmoid(logits)@self.label_graph)

def layer_list(model):
    b=model.features.backbone
    return b.enc_layers if model.cfg['spec']['architecture']=='encoder_decoder' else b.blocks

def unfreeze_last_layers(model,count):
    for layer in list(layer_list(model))[-count:] if count else []:
        layer.requires_grad_(True)
    b=model.features.backbone
    norm=b.ln_enc_f if model.cfg['spec']['architecture']=='encoder_decoder' else (
        b.ln_f if model.cfg['spec']['architecture']=='decoder_only' else b.norm_f)
    norm.requires_grad_(True)

def make_model(cfg):
    spec=cfg['spec']; b,phase1_config,original_parameters=load_phase1(spec['architecture'])
    if cfg['chunk_length']>phase1_config.get('max_len',1024):
        raise ValueError('Chunk length exceeds Phase 1 positional embeddings.')
    selected=None
    if spec['method']=='kd':
        selected=list(range(0,len(b.blocks),2)); b.blocks=nn.ModuleList([b.blocks[i] for i in selected])
    features=CaseFeatures(b,spec['architecture']); features.use_checkpointing=cfg['gradient_checkpointing']
    model=LegalCase(features,cfg)
    if cfg['initialize_label_priors']:
        priors=torch.tensor(np.clip(TRAIN_COUNTS/max(1,len(DATA['train'])),0.005,0.995),dtype=torch.float32)
        with torch.no_grad(): model.classifier[-1].bias.copy_(torch.logit(priors))
    b.requires_grad_(spec['method'] in ['full','kd'])
    targets=[]; method=spec['method']
    if method=='partial': unfreeze_last_layers(model,min(cfg['partial_layers'],len(layer_list(model))))
    elif method=='bitfit':
        for name,p in b.named_parameters(): p.requires_grad_(name.endswith('.bias'))
    elif method=='lorma':
        for name,child in list(b.named_modules()):
            if isinstance(child,nn.Linear) and '.attn.' in name and name.rsplit('.',1)[-1] in ['qkv','proj','q_proj','k_proj','v_proj','o_proj']:
                parent,leaf=name.rsplit('.',1)
                setattr(b.get_submodule(parent),leaf,LowRankMultiplicative(child,cfg['adapter_rank'],cfg['adapter_alpha']))
                targets.append(name)
        assert targets
    elif method=='adapters':
        layers=layer_list(model)
        for i in range(len(layers)):
            layers[i]=ResidualAdapter(layers[i],b.tok_emb.embedding_dim,cfg['bottleneck_dim'])
            targets.append(str(i))
    elif method not in ['full','kd','gradual','linear_probe']:
        raise ValueError('Unknown adaptation method.')
    metadata={'phase1_config':phase1_config,'phase1_parameters':original_parameters,'selected_phase1_layers':selected,
        'adapter_targets':targets,'total_parameters':sum(p.numel() for p in model.parameters()),
        'trainable_parameters_initial':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model,metadata

def forward_batch(model,batch):
    return model(*[batch[k].to(DEVICE) for k in ['input_ids','attention_mask','target_mask','chunk_positions','owners','stats']])

## 8. Multi-label metrics, global/per-label thresholds and output verification

Macro-F1 averages **per-statute positive-class F1**; Micro-F1 aggregates all statute decisions;
Subset Accuracy requires the entire predicted label set to match. Zero-denominator F1 is zero.
The PDF does not supply evaluator code and says label dimension is derived from references.
Default `METRIC_LABEL_SCOPE='reference_range'` assumes IDs `0..max(gold ID)`; known-catalog and
reference-present variants are also reported. Match this control to any published official scorer.

Dev compares 0.5, a bounded global grid, and per-label F1 thresholds shrunk toward the global threshold.
Rare labels below `MIN_DEV_POSITIVES` retain the global threshold. Calibration keeps a global/0.5
fallback if per-label changes hurt Macro-F1. Test labels never calibrate any decision.

In [ ]:
def gold_matrix(split, keys=None):
    return np.stack([encode_labels(DATA[split][key]['label']) for key in (keys or list(DATA[split]))]).astype(np.int64)


def multilabel_metrics(gold, predicted):
    y = np.asarray(gold)
    p = np.asarray(predicted)
    if y.shape != p.shape or y.ndim != 2 or y.shape[1] != NUM_LABELS or not len(y):
        raise ValueError('Expected nonempty case-by-statute matrices.')
    if not np.isin(y, [0, 1]).all() or not np.isin(p, [0, 1]).all():
        raise ValueError('Expected binary multi-hot matrices.')
    y = y.astype(np.int64); p = p.astype(np.int64)
    tp = (y * p).sum(0); fp = ((1 - y) * p).sum(0); fn = (y * (1 - p)).sum(0)
    support = y.sum(0)
    f1 = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(NUM_LABELS), where=(2 * tp + fp + fn) > 0)
    precision = np.divide(tp, tp + fp, out=np.zeros(NUM_LABELS), where=tp + fp > 0)
    recall = np.divide(tp, tp + fn, out=np.zeros(NUM_LABELS), where=tp + fn > 0)
    present = support > 0
    reference_max = max((label for label, n in zip(LABEL_IDS, support) if n), default=-1)
    known_macro = float(f1.mean())
    present_macro = float(f1[present].mean()) if present.any() else 0.0
    range_macro = float(sum(v for label, v in zip(LABEL_IDS, f1) if label <= reference_max) / (reference_max + 1)) if reference_max >= 0 else 0.0
    primary = {'known': known_macro, 'reference_present': present_macro, 'reference_range': range_macro}[METRIC_LABEL_SCOPE]
    mask = np.ones(NUM_LABELS, dtype=bool)
    if METRIC_LABEL_SCOPE == 'reference_present':
        mask = present
    elif METRIC_LABEL_SCOPE == 'reference_range':
        mask = np.asarray([label <= reference_max for label in LABEL_IDS])
    total_tp = int(tp[mask].sum()); total_fp = int(fp[mask].sum()); total_fn = int(fn[mask].sum())
    denominator = 2 * total_tp + total_fp + total_fn
    known_denominator = int(2 * tp.sum() + fp.sum() + fn.sum())
    return {'macro_f1': primary, 'macro_f1_known': known_macro, 'macro_f1_reference_present': present_macro,
            'macro_f1_reference_range': range_macro, 'metric_label_scope': METRIC_LABEL_SCOPE,
            'micro_f1': 2 * total_tp / denominator if denominator else 0.0,
            'subset_accuracy': float(np.all(y[:, mask] == p[:, mask], axis=1).mean()),
            'micro_f1_known': float(2 * tp.sum() / known_denominator) if known_denominator else 0.0,
            'subset_accuracy_known': float(np.all(y == p, axis=1).mean()),
            'precision': total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0,
            'recall': total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0,
            'cases': len(y), 'mean_predicted_labels': float(p.sum(1).mean()),
            'per_label': {str(label): {'precision': float(pr), 'recall': float(re), 'f1': float(f),
                          'support': int(s), 'train_support': int(t), 'tp': int(a), 'fp': int(b), 'fn': int(c)}
                          for label, pr, re, f, s, t, a, b, c in zip(LABEL_IDS, precision, recall, f1, support, TRAIN_COUNTS, tp, fp, fn)}}


def decisions(probabilities, policy):
    p = np.asarray(probabilities, dtype=np.float64)
    if p.ndim != 2 or p.shape[1] != NUM_LABELS or not np.isfinite(p).all() or not ((0 <= p) & (p <= 1)).all():
        raise ValueError('Invalid case-by-statute probabilities.')
    thresholds = np.asarray(policy['thresholds'], dtype=float)
    if thresholds.shape != (NUM_LABELS,) or not np.isfinite(thresholds).all():
        raise ValueError('Invalid threshold vector.')
    selected = p >= thresholds
    if not policy['allow_empty']:
        empty = ~selected.any(1)
        selected[np.flatnonzero(empty), p[empty].argmax(1)] = True
    return selected.astype(np.int64)


def exact_label_threshold(p, y):
    # Positive-class F1 for a single statute; this is not binary outcome Macro-F1.
    p = np.asarray(p, dtype=float); y = np.asarray(y, dtype=np.int64)
    if y.sum() == 0:
        return 0.5
    order = np.argsort(-p, kind='stable'); ps = p[order]; ys = y[order]
    ends = np.flatnonzero(np.r_[ps[:-1] != ps[1:], True])
    tp = np.cumsum(ys)[ends]; count = ends + 1
    scores = 2 * tp / (int(y.sum()) + count)
    best = scores.max()
    options = [(abs(float(t) - 0.5), float(t)) for t, s in zip(ps[ends], scores) if abs(float(s) - best) < 1e-12]
    # Retain 0.5 on a tie rather than using an arbitrary boundary.
    baseline_pred = p >= 0.5
    base_den = int(y.sum()) + int(baseline_pred.sum())
    base_score = 2 * int((baseline_pred * y).sum()) / base_den if base_den else 0.0
    if abs(base_score - best) < 1e-12:
        options.append((0.0, 0.5))
    return min(options)[1]


def optimal_threshold(probabilities, gold, tune=True, cfg=None):
    cfg = cfg or {'threshold_mode': THRESHOLD_MODE, 'threshold_grid': THRESHOLD_GRID,
                  'min_dev_positives': MIN_DEV_POSITIVES, 'threshold_shrinkage': THRESHOLD_SHRINKAGE,
                  'allow_empty': ALLOW_EMPTY_PREDICTIONS}
    p = np.asarray(probabilities, dtype=float); y = np.asarray(gold, dtype=np.int64)
    baseline_policy = {'mode': 'fixed_0.5', 'thresholds': [0.5] * NUM_LABELS, 'allow_empty': cfg['allow_empty']}
    baseline = multilabel_metrics(y, decisions(p, baseline_policy))
    if not tune:
        return baseline_policy, baseline, baseline
    # Compute candidate scores from confusion counts, avoiding per-label JSON reports per grid point.
    def score(policy):
        pred = decisions(p, policy)
        tp = (pred * y).sum(0); den = y.sum(0) + pred.sum(0)
        f = np.divide(2 * tp, den, out=np.zeros(NUM_LABELS), where=den > 0)
        support = y.sum(0)
        if METRIC_LABEL_SCOPE == 'reference_present':
            return float(f[support > 0].mean()) if support.any() else 0.0
        if METRIC_LABEL_SCOPE == 'reference_range':
            maximum = max((label for label, n in zip(LABEL_IDS, support) if n), default=-1)
            return float(sum(v for label, v in zip(LABEL_IDS, f) if label <= maximum) / (maximum + 1)) if maximum >= 0 else 0.0
        return float(f.mean())
    best = baseline_policy; best_score = baseline['macro_f1']
    for threshold in sorted(set(cfg['threshold_grid']), key=lambda t: (abs(t - 0.5), t)):
        policy = {'mode': 'global', 'thresholds': [float(threshold)] * NUM_LABELS, 'allow_empty': cfg['allow_empty']}
        value = score(policy)
        if value > best_score + 1e-12:
            best = policy; best_score = value
    global_policy = copy.deepcopy(best)
    if cfg['threshold_mode'] in ('auto', 'per_label'):
        support = y.sum(0); thresholds = list(global_policy['thresholds'])
        for j in range(NUM_LABELS):
            if support[j] >= cfg['min_dev_positives']:
                local = exact_label_threshold(p[:, j], y[:, j])
                weight = support[j] / (support[j] + cfg['threshold_shrinkage'])
                thresholds[j] = float(weight * local + (1 - weight) * thresholds[j])
        policy = {'mode': 'shrunk_per_label', 'thresholds': thresholds, 'allow_empty': cfg['allow_empty']}
        value = score(policy)
        if value > best_score + 1e-12:
            best = policy; best_score = value
    selected = multilabel_metrics(y, decisions(p, best))
    assert selected['macro_f1'] + 1e-12 >= baseline['macro_f1']
    return best, selected, baseline


def show_scores(metrics, title):
    print(title)
    print(f"Macro-F1 [PRIMARY, {metrics['metric_label_scope']}]: {metrics['macro_f1']:.6f}")
    print(f"Micro-F1: {metrics['micro_f1']:.6f}; Subset Accuracy: {metrics['subset_accuracy']:.6f}")
    print(f"Micro precision/recall: {metrics['precision']:.6f}/{metrics['recall']:.6f}")
    print('Macro-F1 conventions:', {k: metrics[k] for k in ('macro_f1_reference_range', 'macro_f1_reference_present', 'macro_f1_known')})


def probability_vector(logits):
    return torch.sigmoid(logits.double()).numpy()


def make_prediction_rows(split, keys, probabilities, policy):
    chosen = decisions(probabilities, policy)
    return [{'id': DATA[split][key]['id'], 'pred': [int(label) for label, yes in zip(LABEL_IDS, row) if yes]}
            for key, row in zip(keys, chosen)]


def validate_submission(predictions, docs):
    if not isinstance(predictions, list) or len(predictions) != len(docs):
        raise ValueError('Expected exactly one prediction per input instance.')
    expected = {}
    for key, doc in docs.items():
        expected.setdefault((type(doc['id']), doc['id']), []).append(key)
    consumed = Counter(); matched = {}
    for row in predictions:
        if not isinstance(row, dict) or set(row) != {'id', 'pred'}:
            raise ValueError('Expected exactly id and pred fields.')
        case_key(row['id']); identity = (type(row['id']), row['id']); occurrence = consumed[identity]
        if identity not in expected or occurrence >= len(expected[identity]):
            raise ValueError('Unknown, duplicate or changed original case ID.')
        labels = row['pred']
        if not isinstance(labels, list) or any(type(x) is not int or x not in LABEL_TO_INDEX for x in labels):
            raise ValueError('pred must contain valid integer statute IDs only.')
        if len(labels) != len(set(labels)):
            raise ValueError('Duplicate statute IDs within a prediction.')
        matched[expected[identity][occurrence]] = labels; consumed[identity] += 1
    if set(matched) != set(docs):
        raise ValueError('Missing prediction instances.')
    return matched


def save_predictions(output, probabilities, policy):
    rows = make_prediction_rows('test', list(DATA['test']), probabilities, policy)
    validate_submission(rows, DATA['test']); save_json(rows, output / 'prediction.json')
    assert json.loads((output / 'prediction.json').read_text()) == rows
    temporary = output / 'submission.zip.tmp'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
        archive.write(output / 'prediction.json', arcname='prediction.json')
    os.replace(temporary, output / 'submission.zip')
    with zipfile.ZipFile(output / 'submission.zip') as archive:
        assert archive.namelist() == ['prediction.json']
        validate_submission(json.loads(archive.read('prediction.json')), DATA['test'])
        assert json.loads(archive.read('prediction.json')) == rows
    return rows


def save_probabilities(output, split, keys, p):
    save_json({'label_ids': LABEL_IDS, 'rows': [{'id': DATA[split][key]['id'], 'instance_key': key,
              'probabilities': [float(v) for v in values]} for key, values in zip(keys, p)]},
              output / (split + '_probabilities.json'))


def load_probability_file(path, split):
    saved = json.loads((path / (split + '_probabilities.json')).read_text())
    if saved['label_ids'] != LABEL_IDS:
        raise ValueError('Probability label ordering differs.')
    rows = saved['rows']; mapping = {row['instance_key']: row for row in rows}
    if len(mapping) != len(rows) or set(mapping) != set(DATA[split]):
        raise ValueError('Probability instance keys differ.')
    for key, row in mapping.items():
        original = DATA[split][key]['id']
        if type(row['id']) is not type(original) or row['id'] != original:
            raise ValueError('Probability cache original ID changed.')
    values = np.asarray([mapping[key]['probabilities'] for key in DATA[split]], dtype=float)
    if values.shape != (len(DATA[split]), NUM_LABELS) or not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
        raise ValueError('Invalid saved probabilities.')
    return values

@torch.no_grad()
def predict_logits(model,split,cfg,description=None):
    dataset=CaseDataset(split,cfg,training=False)
    loader=DataLoader(dataset,batch_size=cfg['batch_size'],shuffle=False,collate_fn=collate_cases,
        num_workers=NUM_WORKERS,generator=torch.Generator().manual_seed(cfg['seed']))
    was_training=model.training; model.eval(); result={}
    for batch in tqdm(loader,desc=description or 'Predicting '+split,unit='batch'):
        with amp_context(): logits=forward_batch(model,batch)
        logits=logits.float().cpu()
        if not torch.isfinite(logits).all(): raise ValueError('Non-finite case logits.')
        for i,key in enumerate(batch['keys']):
            if key in result: raise ValueError('Duplicate prediction case.')
            result[key]=logits[i]
    model.train(was_training)
    if set(result)!=set(DATA[split]): raise ValueError('Missing/extra prediction IDs.')
    keys=list(DATA[split]); array=torch.stack([result[k] for k in keys])
    return keys,array

## 9. Full configuration fingerprints, larger-teacher cache and deterministic resume

In [ ]:
def capture_rng():
    return {'python': random.getstate(), 'numpy': {
        'kind': np.random.get_state()[0], 'keys': np.random.get_state()[1].tolist(),
        'pos': np.random.get_state()[2], 'gauss': np.random.get_state()[3],
        'cached': np.random.get_state()[4]}, 'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else []}

In [ ]:
def restore_rng(state):
    random.setstate(state['python'])
    n = state['numpy']
    np.random.set_state((n['kind'], np.array(n['keys'], dtype=np.uint32), n['pos'], n['gauss'], n['cached']))
    torch.set_rng_state(state['torch'])
    if DEVICE.type == 'cuda' and state['cuda']:
        torch.cuda.set_rng_state_all(state['cuda'])

In [ ]:
def base_config(name,overrides=None):
    cfg={'experiment':name,'spec':copy.deepcopy(EXPERIMENTS[name]),'run_tag':RUN_TAG,'seed':SEED,
        'epochs':EPOCHS,'batch_size':BATCH_SIZE,'grad_accum':GRAD_ACCUM_STEPS,'chunk_length':CHUNK_LENGTH,
        'chunk_overlap':CHUNK_OVERLAP,'max_chunks':MAX_CHUNKS,'chunk_batch_size':CHUNK_BATCH_SIZE,
        'train_chunk_policy':TRAIN_CHUNK_POLICY,'eval_chunk_policy':EVAL_CHUNK_POLICY,'pooling':DOC_POOLING,
        'doc_hidden':DOC_HIDDEN,'dropout':HEAD_DROPOUT,'head_lr_multiplier':HEAD_LR_MULTIPLIER,
        'use_class_weights':USE_CLASS_WEIGHTS,'class_weight_power':CLASS_WEIGHT_POWER,'weight_decay':WEIGHT_DECAY,
        'warmup_fraction':WARMUP_FRACTION,'patience':EARLY_STOPPING_PATIENCE,'gradient_checkpointing':GRADIENT_CHECKPOINTING,
        'adapter_rank':ADAPTER_RANK,'adapter_alpha':ADAPTER_ALPHA,'bottleneck_dim':BOTTLENECK_DIM,
        'partial_layers':PARTIAL_LAYERS,'unfreeze_layers_per_epoch':UNFREEZE_LAYERS_PER_EPOCH,
        'kd_weight':KD_WEIGHT,'kd_temperature':KD_TEMPERATURE,'teacher_requested':TEACHER_RUN_DIR,
        'max_optimizer_steps':MAX_OPTIMIZER_STEPS,'tune_threshold':TUNE_THRESHOLD,
        'phase1_heads':PHASE1_HEADS,'tokenizer_hash':TOKENIZER_HASH,'data_fingerprint':DATA_FINGERPRINT,
        'num_labels':NUM_LABELS,'label_ids':LABEL_IDS,'metric_label_scope':METRIC_LABEL_SCOPE,
        'loss_name':LOSS_NAME,'pos_weight_cap':POS_WEIGHT_CAP,'focal_gamma':FOCAL_GAMMA,
        'asl_gamma_pos':ASL_GAMMA_POS,'asl_gamma_neg':ASL_GAMMA_NEG,'asl_clip':ASL_CLIP,
        'label_smoothing':LABEL_SMOOTHING,'initialize_label_priors':INITIALIZE_LABEL_PRIORS,
        'label_graph_weight':0.0,'threshold_mode':THRESHOLD_MODE,'threshold_grid':THRESHOLD_GRID,
        'min_dev_positives':MIN_DEV_POSITIVES,'threshold_shrinkage':THRESHOLD_SHRINKAGE,
        'allow_empty':ALLOW_EMPTY_PREDICTIONS}
    for key,value in (overrides or {}).items():
        if key=='lr': cfg['spec']['lr']=value
        elif key in cfg: cfg[key]=value
        else: raise ValueError('Unknown tuning override '+key)
    assert 4<=cfg['chunk_length']<=1024 and 0<=cfg['chunk_overlap']<cfg['chunk_length']-2
    assert cfg['max_chunks']>=0 and cfg['chunk_batch_size']>0 and cfg['doc_hidden']>0
    assert cfg['pooling'] in ['attention','mean','max','mean_max','transformer']
    assert cfg['pooling']!='transformer' or cfg['doc_hidden']%4==0
    assert cfg['train_chunk_policy'] in ['spread','head','tail','head_tail','random_mix','all']
    assert cfg['eval_chunk_policy'] in ['spread','head','tail','head_tail','all']
    assert cfg['partial_layers']>0 and cfg['unfreeze_layers_per_epoch']>0 and cfg['adapter_rank']>0
    assert cfg['loss_name'] in ['bce','focal','asymmetric']
    assert cfg['threshold_mode'] in ['global','per_label','auto']
    assert 0<=cfg['label_smoothing']<1 and cfg['pos_weight_cap']>=1
    assert cfg['threshold_shrinkage']>=0 and cfg['min_dev_positives']>=1
    assert cfg['focal_gamma']>=0 and cfg['asl_gamma_pos']>=0 and cfg['asl_gamma_neg']>=0 and 0<=cfg['asl_clip']<1
    assert cfg['num_labels']==NUM_LABELS and cfg['label_ids']==LABEL_IDS
    assert cfg['metric_label_scope']==METRIC_LABEL_SCOPE
    return cfg

def run_path(run_name): return PROJECT_DIR/(run_name+'_'+RUN_TAG)

def coverage_report(cfg,output):
    report={}
    for split in DATA:
        dataset=CaseDataset(split,cfg,False); ratios=[]; counts=[]
        for i in tqdm(range(len(dataset)),desc='Auditing chunk coverage: '+split,unit='case'):
            row=dataset[i]; ratios.append(row['coverage']); counts.append(len(row['chunks']))
        report[split]={'cases':len(dataset),'mean_token_coverage':float(np.mean(ratios)),
            'cases_with_full_coverage':sum(r>=1-1e-9 for r in ratios),'mean_chunks':float(np.mean(counts)),
            'max_chunks':max(counts)}
    save_json(report,output/'chunk_coverage.json'); print('Chunk coverage:',report)
    return report

def teacher_logits_for_student(student_parameters,cfg):
    choices=[]
    paths=[Path(cfg['teacher_requested'])] if cfg['teacher_requested'] else [p.parent for p in PROJECT_DIR.glob('*/metrics.json')]
    for path in paths:
        if not (path/'best_model.pt').is_file() or not (path/'run_config.json').is_file(): continue
        metrics=json.loads((path/'metrics.json').read_text()); saved_cfg=json.loads((path/'run_config.json').read_text())
        if saved_cfg['data_fingerprint']!=DATA_FINGERPRINT or saved_cfg['tokenizer_hash']!=TOKENIZER_HASH: continue
        if not metrics['training_complete'] or metrics['total_parameters']<=student_parameters: continue
        if saved_cfg['spec']['method']=='kd': continue
        choices.append((metrics['macro_f1'],str(path)))
    if not choices: raise FileNotFoundError('KD requires a completed, compatible, larger Phase 5 teacher. Run a core non-KD experiment first.')
    _,selected=max(choices); path=Path(selected)
    teacher_cfg=json.loads((path/'run_config.json').read_text())
    checkpoint_stat=(path/'best_model.pt').stat()
    signature={'checkpoint_bytes':checkpoint_stat.st_size,'checkpoint_mtime_ns':checkpoint_stat.st_mtime_ns,
               'data_fingerprint':DATA_FINGERPRINT,'tokenizer_hash':TOKENIZER_HASH}
    cache_path=path/'teacher_train_logits.pt'
    cached=torch_load(cache_path) if cache_path.is_file() else None
    if cached is None or cached.get('signature')!=signature:
        if teacher_cfg['phase1_heads']!=PHASE1_HEADS: raise ValueError('Teacher Phase 1 head counts differ.')
        state=torch_load(path/'best_model.pt')
        teacher,_=make_model(teacher_cfg); teacher.load_state_dict(state['model'],strict=True)
        del state
        teacher.eval().requires_grad_(False); teacher=teacher.to(DEVICE)
        keys,logits=predict_logits(teacher,'train',teacher_cfg,'Caching teacher logits on TRAIN only')
        cached={'signature':signature,'keys':keys,'logits':logits}
        save_torch(cached,cache_path)
        del teacher; gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    keys=cached['keys']; logits=cached['logits']
    if set(keys)!=set(DATA['train']) or len(keys)!=len(set(keys)) or logits.shape!=(len(keys),NUM_LABELS) or not torch.isfinite(logits).all():
        raise ValueError('Invalid teacher train-logit cache.')
    print('KD teacher:',path,'(student may use different chunking/hyperparameters)')
    return {key:logits[i] for i,key in enumerate(keys)},selected,signature

## 10. Preflight checks: multi-label behavior, thresholds, windows and LoRMA

These checks establish implementation behavior, not expected leaderboard performance.
Full post-training checks independently reload weights and recompute every reported dev metric.

In [ ]:
assert NUM_LABELS>0 and all(type(label) is int for label in LABEL_IDS)
y=np.eye(NUM_LABELS,dtype=np.int64)
assert multilabel_metrics(y,y)['micro_f1']==1.0
assert multilabel_metrics(y,y)['subset_accuracy']==1.0
assert multilabel_metrics(y,np.zeros_like(y))['micro_f1']==0.0
p=0.1+0.8*y
policy,selected,baseline=optimal_threshold(p,y)
assert selected['macro_f1']+1e-12>=baseline['macro_f1']
if NUM_LABELS>1:
    assert decisions(np.full((1,NUM_LABELS),0.9),{'thresholds':[0.5]*NUM_LABELS,'allow_empty':True}).sum()==NUM_LABELS
for length in [0,1,500,2000]:
    ranges=window_ranges(length,384,64)
    assert covered_tokens(ranges)==length
base=nn.Linear(7,11); layer=LowRankMultiplicative(copy.deepcopy(base),3,6)
x=torch.randn(2,5,7); torch.testing.assert_close(layer(x),base(x))
layer(x).sum().backward(); assert layer.B.weight.grad.abs().sum()>0 and layer.base.weight.grad is None
print('PASS: multi-label metric invariants, threshold baseline, chunk coverage and LoRMA identity/gradients.')

## 11. Train, calibrate on dev, reload the best checkpoint and verify every export

Independent multi-label BCE, focal or asymmetric loss with train-only positive weights.
KD uses temperature-scaled independent Bernoulli targets, not mutually exclusive softmax targets.
The best dev-calibrated checkpoint is reloaded and its full metrics must match. JSON predictions
are decoded back to multi-hot vectors and scored again. Every test ID and ZIP entry is verified.
Progress bars cover tokenization, coverage, epochs, teacher prediction, dev/test and sweeps.

In [ ]:
def positive_weights(cfg):
    counts = TRAIN_COUNTS.astype(float)
    if not cfg['use_class_weights']:
        return None
    weights = ((len(DATA['train']) - counts) / np.maximum(counts, 1)) ** cfg['class_weight_power']
    # No artificial learning signal is created for statutes with no positive train examples.
    weights[counts == 0] = 1.0
    return torch.tensor(np.clip(weights, 1.0, cfg['pos_weight_cap']), dtype=torch.float32, device=DEVICE)


def multilabel_loss(logits, targets, weights, cfg):
    x = logits.float(); y = targets.float()
    smooth = cfg['label_smoothing']; y = y * (1 - smooth) + 0.5 * smooth
    if cfg['loss_name'] in ('bce', 'focal'):
        loss = F.binary_cross_entropy_with_logits(x, y, pos_weight=weights, reduction='none')
        if cfg['loss_name'] == 'focal':
            p = x.sigmoid(); pt = y * p + (1 - y) * (1 - p)
            loss = loss * (1 - pt).pow(cfg['focal_gamma'])
    elif cfg['loss_name'] == 'asymmetric':
        p = x.sigmoid(); negative = (1 - p + cfg['asl_clip']).clamp(max=1)
        positive_weight = weights if weights is not None else 1.0
        loss = -positive_weight * y * torch.log(p.clamp_min(1e-8)) * (1 - p).pow(cfg['asl_gamma_pos'])
        loss = loss - (1 - y) * torch.log(negative.clamp_min(1e-8)) * (1 - negative).pow(cfg['asl_gamma_neg'])
    else:
        raise ValueError('Unknown multi-label loss.')
    return loss.mean()


def distillation_loss(student, teacher, temperature):
    # Independent Bernoulli labels; softmax/KL would incorrectly make statutes mutually exclusive.
    return F.binary_cross_entropy_with_logits(student.float() / temperature,
        torch.sigmoid(teacher.float() / temperature)) * temperature ** 2


def save_error_analysis(output, keys, probabilities, policy, metrics):
    import csv
    predicted = decisions(probabilities, policy)
    rows = []
    for key, values, chosen in zip(keys, probabilities, predicted):
        gold = set(DATA['dev'][key]['label']); pred = {label for label, yes in zip(LABEL_IDS, chosen) if yes}
        if pred != gold:
            rows.append({'id': DATA['dev'][key]['id'], 'instance_key': key, 'gold': sorted(gold),
                         'pred': sorted(pred), 'missed': sorted(gold - pred), 'extra': sorted(pred - gold)})
    save_json(rows, output / 'dev_errors.json')
    with (output / 'per_label_metrics.csv').open('w', newline='', encoding='utf-8') as stream:
        columns = ['statute_id', 'train_support', 'support', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn']
        writer = csv.DictWriter(stream, fieldnames=columns); writer.writeheader()
        for label, values in metrics['per_label'].items():
            writer.writerow({'statute_id': label, **values})
    rare = [row for row in metrics['per_label'].values() if row['train_support'] <= 10 and row['support'] > 0]
    save_json({'rare_definition': 'train support <=10 and present in dev', 'rare_labels': len(rare),
               'rare_macro_f1': float(np.mean([row['f1'] for row in rare])) if rare else None,
               'mean_gold_labels': float(np.mean([len(DATA['dev'][key]['label']) for key in keys])),
               'mean_predicted_labels': metrics['mean_predicted_labels']}, output / 'rare_label_analysis.json')
    missed = sorted(metrics['per_label'].items(), key=lambda pair: pair[1]['fn'], reverse=True)[:20]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar([label for label, row in missed], [row['fn'] for label, row in missed])
    ax.set(xlabel='Statute ID', ylabel='False negatives on DEV', title='Most missed statutes')
    fig.tight_layout(); fig.savefig(output / 'missed_statutes.png', dpi=130); plt.show(); plt.close(fig)

In [ ]:
def optimizer_groups(model,cfg):
    groups={}
    for name,p in model.named_parameters():
        if not p.requires_grad and cfg['spec']['method']!='gradual': continue
        head=not name.startswith('features.backbone.')
        no_decay=p.ndim<2 or name.endswith('.bias') or 'norm' in name or '.ln' in name
        groups.setdefault((head,no_decay),[]).append(p)
    return [{'params':params,'lr':cfg['spec']['lr']*(cfg['head_lr_multiplier'] if head else 1),
        'weight_decay':0.0 if no_decay else cfg['weight_decay']} for (head,no_decay),params in groups.items()]

def run_experiment(name,overrides=None,run_name=None):
    cfg=base_config(name,overrides); seed_all(cfg['seed']); run_name=run_name or name
    output=run_path(run_name); output.mkdir(parents=True,exist_ok=True)
    config_path=output/'run_config.json'
    if config_path.is_file() and json.loads(config_path.read_text())!=cfg:
        raise ValueError('Settings changed for '+str(output)+'. Choose a new RUN_TAG or trial name.')
    if not RESUME and (output/'last.pt').exists(): raise ValueError('Use a fresh tag instead of overwriting a run.')
    save_json(cfg,config_path); coverage_report(cfg,output)
    # Obtain teacher first so its temporary GPU model is released before moving the student to GPU.
    model,metadata=make_model(cfg); teacher_cache=None; teacher_name=None; teacher_signature=None
    if cfg['spec']['method']=='kd':
        student_rng=capture_rng()
        try:
            teacher_cache,teacher_name,teacher_signature=teacher_logits_for_student(metadata['total_parameters'],cfg)
        finally:
            restore_rng(student_rng)
    metadata['teacher_run']=teacher_name; metadata['teacher_signature']=teacher_signature; model=model.to(DEVICE)
    save_json(metadata,output/'model_metadata.json')
    print(run_name,'parameters:',metadata['total_parameters'],'initial trainable:',metadata['trainable_parameters_initial'])
    optimizer=torch.optim.AdamW(optimizer_groups(model,cfg))
    weights=positive_weights(cfg)
    save_json({'label_ids':LABEL_IDS,'counts':TRAIN_COUNTS.tolist(),
               'weights':weights.tolist() if weights is not None else [1.0]*NUM_LABELS},output/'class_weights.json')
    dataset=CaseDataset('train',cfg,training=True)
    n_batches=math.ceil(len(dataset)/cfg['batch_size']); total_steps=cfg['epochs']*math.ceil(n_batches/cfg['grad_accum'])
    warmup=max(1,round(total_steps*cfg['warmup_fraction']))
    def scale(step):
        return (step+1)/warmup if step<warmup else 0.5*(1+math.cos(math.pi*min(1,(step-warmup)/max(1,total_steps-warmup))))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,scale)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda' and AMP_DTYPE==torch.float16)
    history=[]; best_f1=-1.0; bad_epochs=0; step=0; start_epoch=0; start_batch=0; complete=False; elapsed=0.0
    if RESUME and (output/'last.pt').is_file():
        state=torch_load(output/'last.pt')
        if state['run_config']!=cfg or state['metadata']['teacher_run']!=teacher_name or state['metadata'].get('teacher_signature')!=teacher_signature: raise ValueError('Saved settings/teacher differ.')
        model.load_state_dict(state['model'],strict=True); optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler']); restore_rng(state['rng'])
        history=state['history']; best_f1=state['best_f1']; bad_epochs=state['bad_epochs']; step=state['step']
        start_epoch=state['epoch']; start_batch=state['next_batch']; complete=state['training_complete']; elapsed=state['training_seconds']
        del state; print('Resume epoch',start_epoch+1,'batch',start_batch,'step',step,'complete',complete)
    started=time.time()
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
    def persist(epoch,next_batch):
        save_torch({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
            'scaler':scaler.state_dict(),'rng':capture_rng(),'run_config':cfg,'metadata':metadata,'history':history,
            'best_f1':best_f1,'bad_epochs':bad_epochs,'step':step,'epoch':epoch,'next_batch':next_batch,
            'training_complete':complete,'training_seconds':elapsed+time.time()-started},output/'last.pt')
        save_json(history,output/'history.json')
    if not complete:
        for epoch in range(start_epoch,cfg['epochs']):
            if cfg['max_optimizer_steps'] is not None and step>=cfg['max_optimizer_steps']: break
            if cfg['spec']['method']=='gradual':
                unfreeze_last_layers(model,min(len(layer_list(model)),(epoch+1)*cfg['unfreeze_layers_per_epoch']))
            dataset.epoch=epoch
            order=list(range(len(dataset))); random.Random(cfg['seed']+epoch).shuffle(order)
            begin=start_batch if epoch==start_epoch else 0
            loader=DataLoader(torch.utils.data.Subset(dataset,order[begin*cfg['batch_size']:]),batch_size=cfg['batch_size'],
                shuffle=False,collate_fn=collate_cases,num_workers=NUM_WORKERS,generator=torch.Generator().manual_seed(cfg['seed']+epoch))
            model.train(); optimizer.zero_grad(set_to_none=True); group_loss=0.0; next_batch=begin; capped=False
            progress=tqdm(loader,desc=f'{run_name} epoch {epoch+1}/{cfg["epochs"]}',unit='case batch')
            for local_i,batch in enumerate(progress):
                i=begin+local_i; next_batch=i+1
                group_start=(i//cfg['grad_accum'])*cfg['grad_accum']
                group_cases=min(cfg['batch_size']*cfg['grad_accum'],len(dataset)-group_start*cfg['batch_size'])
                case_fraction=len(batch['keys'])/group_cases
                labels=batch['labels'].to(DEVICE)
                with amp_context():
                    logits=forward_batch(model,batch)
                    loss=multilabel_loss(logits,labels,weights,cfg)
                    if teacher_cache is not None:
                        teacher_logits=torch.stack([teacher_cache[k] for k in batch['keys']]).to(DEVICE)
                        temperature=cfg['kd_temperature']
                        kd=distillation_loss(logits,teacher_logits,temperature)
                        loss=(1-cfg['kd_weight'])*loss+cfg['kd_weight']*kd
                if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss; latest saved checkpoint is preserved.')
                scaler.scale(loss*case_fraction).backward(); group_loss+=loss.detach().item()*case_fraction
                progress.set_postfix(loss=f'{loss.item():.4f}',step=step,lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
                if next_batch%cfg['grad_accum']==0 or next_batch==n_batches:
                    scaler.unscale_(optimizer); nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
                    previous=scaler.get_scale(); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                    if scaler.get_scale()>=previous:
                        scheduler.step(); step+=1
                        history.append({'kind':'train','epoch':epoch+1,'step':step,'loss':group_loss,'lr':optimizer.param_groups[0]['lr']})
                        if step%SAVE_EVERY_STEPS==0: persist(epoch,next_batch)
                    group_loss=0.0
                    if cfg['max_optimizer_steps'] is not None and step>=cfg['max_optimizer_steps']: capped=True; break
            progress.close()
            keys,dev_logits=predict_logits(model,'dev',cfg)
            p=probability_vector(dev_logits); gold=gold_matrix('dev',keys)
            threshold,metrics,baseline=optimal_threshold(p,gold,cfg['tune_threshold'],cfg)
            history.append({'kind':'dev','epoch':epoch+1,'step':step,'macro_f1':metrics['macro_f1'],
                'precision':metrics['precision'],'recall':metrics['recall'],'baseline_macro_f1':baseline['macro_f1'],'threshold':threshold})
            print(f"Dev macro-F1={metrics['macro_f1']:.6f}; precision={metrics['precision']:.6f}; recall={metrics['recall']:.6f}; threshold_strategy={threshold['mode']}")
            if metrics['macro_f1']>best_f1:
                best_f1=metrics['macro_f1']; bad_epochs=0
                save_torch({'model':model.state_dict(),'run_config':cfg,'metadata':metadata,'epoch':epoch+1,'step':step},output/'best_model.pt')
                save_json({'threshold':threshold,'metrics':metrics,'baseline_metrics':baseline},output/'best_dev_metrics.json')
            else: bad_epochs+=1
            finished=next_batch==n_batches
            early=finished and cfg['patience']>0 and bad_epochs>=cfg['patience']
            complete=finished and (epoch+1==cfg['epochs'] or early)
            persist(epoch+1 if finished else epoch,0 if finished else next_batch)
            if complete or capped: break
    if not (output/'best_model.pt').is_file(): raise RuntimeError('No evaluated checkpoint: increase the smoke-test cap or train normally.')
    state=torch_load(output/'best_model.pt'); model.load_state_dict(state['model'],strict=True)
    best_epoch=state['epoch']; del state
    keys,dev_logits=predict_logits(model,'dev',cfg,'Verifying reloaded best checkpoint on DEV')
    p=probability_vector(dev_logits); gold=gold_matrix('dev',keys)
    threshold,metrics,baseline=optimal_threshold(p,gold,cfg['tune_threshold'],cfg)
    recorded=json.loads((output/'best_dev_metrics.json').read_text())
    assert metrics==recorded['metrics'], 'Reloaded checkpoint metrics differ from the selected epoch.'
    assert baseline==recorded['baseline_metrics'], 'Reloaded 0.5-threshold metrics differ.'
    save_probabilities(output,'dev',keys,p)
    dev_predictions=make_prediction_rows('dev',keys,p,threshold)
    validate_submission(dev_predictions,DATA['dev']); save_json(dev_predictions,output/'dev_prediction.json')
    reloaded=json.loads((output/'dev_prediction.json').read_text())
    reloaded_map=validate_submission(reloaded,DATA['dev'])
    assert multilabel_metrics(gold,np.stack([encode_labels(reloaded_map[key]) for key in keys]))==metrics
    save_error_analysis(output,keys,p,threshold,metrics)
    metrics.update({'experiment':name,'run_name':run_name,'architecture':cfg['spec']['architecture'],'method':cfg['spec']['method'],
        'threshold':threshold,'baseline_macro_f1':baseline['macro_f1'],'training_complete':complete,'optimizer_steps':step,
        'best_epoch':best_epoch,'trainable_parameters_final':sum(p.numel() for p in model.parameters() if p.requires_grad),
        'training_seconds':elapsed+time.time()-started,'peak_gpu_memory_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else 0,**metadata})
    save_json(metrics,output/'metrics.json'); save_json({'threshold':threshold,'rule':'independent statute probabilities >= per-label thresholds'},output/'threshold.json')
    show_scores(metrics,'Post-training DEV — reloaded best checkpoint')
    tr=[r for r in history if r['kind']=='train']; dv=[r for r in history if r['kind']=='dev']
    fig,axes=plt.subplots(1,2,figsize=(12,4)); axes[0].plot([r['step'] for r in tr],[r['loss'] for r in tr])
    axes[0].set(xlabel='Optimizer step',ylabel='Training loss')
    axes[1].plot([r['step'] for r in dv],[r['macro_f1'] for r in dv],label='Selected threshold')
    axes[1].plot([r['step'] for r in dv],[r['baseline_macro_f1'] for r in dv],label='Threshold 0.5')
    axes[1].legend(); axes[1].set(xlabel='Optimizer step',ylabel='Dev macro-F1')
    fig.tight_layout(); fig.savefig(output/'training_curves.png',dpi=150); plt.show(); plt.close(fig)
    test_keys,test_logits=predict_logits(model,'test',cfg); test_p=probability_vector(test_logits)
    save_probabilities(output,'test',test_keys,test_p); predictions=save_predictions(output,test_p,threshold)
    verification={'checkpoint_reload_macro_f1_matches':True,'dev_json_metrics_roundtrip':True,
        'one_prediction_per_test_case':True,'original_ids_and_types':True,'unique_valid_integer_statute_sets':True,
        'zip_root_and_json_match':True,'train_dev_test_separate':True,
        'overlap_policy':SPLIT_OVERLAP_POLICY,
        'remaining_train_instances_matching_heldout_text':split_audit['remaining_train_instances_matching_heldout_text'],'test_cases':len(predictions),
        'training_complete':complete,'official_test_score':None,'score_scope':'official DEV used for model and threshold selection'}
    save_json(verification,output/'verification.json'); print('Verification PASS. Saved:',output/'submission.zip')
    if not complete: print('Capped smoke-test artifact; not eligible as a completed experiment/teacher.')
    del model,optimizer,teacher_cache; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return metrics

## 12. Run one experiment, the core set, extras and/or bounded tuning

In [ ]:
selected=list(CORE_EXPERIMENTS) if RUN_ALL else [EXPERIMENT]
if RUN_EXTRA_APPROACHES:
    selected=list(dict.fromkeys(selected+list(EXTRA_EXPERIMENTS)))
results=[]
for name in tqdm(selected,desc='LSI approaches',unit='run'):
    results.append(run_experiment(name))
if RUN_TUNING:
    for trial in tqdm(TUNING_TRIALS,desc='Bounded hyperparameter trials',unit='trial'):
        results.append(run_experiment(TUNE_BASE_EXPERIMENT,trial['overrides'],TUNE_BASE_EXPERIMENT+'__'+trial['name']))

## 13. Optional saved-probability ensemble and export verification

Enable `BUILD_ENSEMBLE` after runs finish. Uses compatible completed runs only, aligns case instances
and label columns, and compares weighted probability pairs to the best single model using dev.
Calibration is repeated on blended probabilities. Test labels are never used. Ensemble is an extra
inference artifact and cannot replace the four architecture-specific adaptation submissions.

In [ ]:
def build_ensemble():
    candidates=[]
    for file in PROJECT_DIR.glob('*/metrics.json'):
        path=file.parent
        if not (path/'run_config.json').is_file() or not (path/'dev_probabilities.json').is_file() or not (path/'test_probabilities.json').is_file(): continue
        cfg=json.loads((path/'run_config.json').read_text()); metrics=json.loads(file.read_text())
        if cfg['data_fingerprint']!=DATA_FINGERPRINT or cfg['tokenizer_hash']!=TOKENIZER_HASH or not metrics['training_complete']: continue
        candidates.append((metrics['macro_f1'],str(path)))
    candidates=sorted(candidates,reverse=True)[:ENSEMBLE_TOP_K]
    if len(candidates)<2:
        print('Ensemble skipped: at least two compatible completed runs are needed.'); return None
    paths=[Path(p) for _,p in candidates]
    dev=[load_probability_file(p,'dev') for p in paths]; test=[load_probability_file(p,'test') for p in paths]
    gold=gold_matrix('dev')
    threshold,metrics,_=optimal_threshold(dev[0],gold,TUNE_THRESHOLD)
    best=(metrics,[str(paths[0])],[1.0],threshold,test[0]); trials=[]
    for a in range(len(paths)):
        for b in range(a+1,len(paths)):
            for weight in ENSEMBLE_PAIR_WEIGHTS:
                blended=weight*dev[a]+(1-weight)*dev[b]
                t,m,_=optimal_threshold(blended,gold,TUNE_THRESHOLD)
                trials.append({'sources':[str(paths[a]),str(paths[b])],'weights':[weight,1-weight],'threshold':t,'macro_f1':m['macro_f1']})
                if m['macro_f1']>best[0]['macro_f1']+1e-12:
                    best=(m,[str(paths[a]),str(paths[b])],[weight,1-weight],t,weight*test[a]+(1-weight)*test[b])
    metrics,sources,weights,threshold,probabilities=best
    output=PROJECT_DIR/('ensemble_'+RUN_TAG); output.mkdir(exist_ok=True)
    save_json({'sources':sources,'weights':weights,'threshold':threshold,'data_fingerprint':DATA_FINGERPRINT,
               'selected_on':'dev','trials':trials},output/'ensemble_recipe.json')
    save_json(metrics,output/'metrics.json'); save_predictions(output,probabilities,threshold)
    show_scores(metrics,'Selected ensemble/single model on DEV'); print('Saved:',output/'submission.zip')
    return metrics

if BUILD_ENSEMBLE: ensemble_metrics=build_ensemble()
for result in results:
    output=run_path(result['run_name'])
    show_scores(json.loads((output/'metrics.json').read_text()),result['run_name'])
    print('Verification:',json.loads((output/'verification.json').read_text()))
    with zipfile.ZipFile(output/'submission.zip') as archive:
        assert archive.namelist()==['prediction.json']
        predictions=json.loads(archive.read('prediction.json'))
    validate_submission(predictions,DATA['test'])

## 14. Where to change what in Colab

All editable controls are in **Section 3**. Start with one complete `decoder_full` run, inspect runtime,
coverage, label frequencies and dev errors, then compare methods. Phase 4 scores do not select Phase 5 winners.

| Goal | Section 3 control |
|---|---|
| Run one profile | `EXPERIMENT='hybrid_lorma'`, `RUN_ALL=False` |
| Six core runs | `RUN_ALL=True` (KD runs last) |
| Seven optional profiles | `RUN_EXTRA_APPROACHES=True` |
| Bounded automatic tuning | `RUN_TUNING=True`, `TUNE_BASE_EXPERIMENT`, edit/shorten `TUNING_TRIALS` |
| Fresh manual comparison | New `RUN_TAG`; every run starts from Phase 1 |
| Exact resumption | Same settings, experiment and tag; `RESUME=True` |
| Training duration | `EPOCHS`, `EARLY_STOPPING_PATIENCE` |
| Learning rate | Change the profile's `lr` in `CORE_EXPERIMENTS`/`EXTRA_EXPERIMENTS` after their definitions |
| Head learning rate | `HEAD_LR_MULTIPLIER` times the profile LR |
| Effective case batch | `BATCH_SIZE * GRAD_ACCUM_STEPS` |
| GPU chunk microbatch | `CHUNK_BATCH_SIZE` (reduce if out of memory) |
| More document coverage | `MAX_CHUNKS=8`; `0` or policy `all` uses every window and costs more |
| Longer windows | `CHUNK_LENGTH`, `CHUNK_OVERLAP`; must fit the Phase 1 positional limit |
| Context selection | `TRAIN_CHUNK_POLICY`, `EVAL_CHUNK_POLICY`: spread/head/tail/head_tail/all; random_mix is train-only |
| Hierarchical pooling | `DOC_POOLING`: attention/mean/max/mean_max/transformer |
| Rare label imbalance | `USE_CLASS_WEIGHTS`, `CLASS_WEIGHT_POWER`, `POS_WEIGHT_CAP` |
| Loss | `LOSS_NAME='bce'`, `'focal'` or `'asymmetric'`; gamma/clip controls nearby |
| Label priors | `INITIALIZE_LABEL_PRIORS`; estimates use train labels only |
| Label relationships | Tuning override `label_graph_weight=0.2`; graph is computed from train co-occurrence only |
| Thresholds | `THRESHOLD_MODE`, `THRESHOLD_GRID`, `MIN_DEV_POSITIVES`, `THRESHOLD_SHRINKAGE` |
| Empty predictions | `ALLOW_EMPTY_PREDICTIONS=False` forces top one only when no statute passes its threshold |
| Official scorer convention | `METRIC_LABEL_SCOPE`; all three Macro-F1 variants are saved |
| Exact-text split overlap | `SPLIT_OVERLAP_POLICY='exclude_train'` removes training copies of dev/test; `error` stops; `report` retains all and audits |
| LoRMA | `ADAPTER_RANK`, `ADAPTER_ALPHA` |
| Residual adapters | `BOTTLENECK_DIM` |
| Partial/gradual tuning | `PARTIAL_LAYERS`, `UNFREEZE_LAYERS_PER_EPOCH` |
| Distillation | `KD_WEIGHT`, `KD_TEMPERATURE`, `TEACHER_RUN_DIR` |
| Ensemble | `BUILD_ENSEMBLE=True`, `ENSEMBLE_TOP_K`, `ENSEMBLE_PAIR_WEIGHTS` |

**Individual parallel sessions:** make separate Colab notebook copies and choose distinct experiments.
Keep `RUN_ALL=False`, `RUN_EXTRA_APPROACHES=False`, `RUN_TUNING=False`, `BUILD_ENSEMBLE=False`.
Prepare Sections 1–5 once before starting other sessions, and do not train the same experiment/tag
in two runtimes. Distinct experiments may share a tag; each writes its own folder. The final summary
and ensemble can be regenerated in one runtime after all training completes.

**KD:** finish a larger Phase 5 teacher first. Leave `TEACHER_RUN_DIR=None` to select the best compatible
completed larger non-KD model by dev Macro-F1, or supply an explicit source folder. The teacher may use
another tag/window budget. Its train logits are cached and checked against checkpoint metadata. The
student starts from alternating Phase 1 lightweight layers, not random or external pretrained weights.

**Tuning really trains each trial.** Overrides are saved in distinct `<base>__<trial>_<tag>` folders.
Trials vary one factor relative to the base, not cumulatively. To combine promising settings, create a
new trial with several overrides or edit the controls under a fresh tag. Per-label thresholds are fitted
inside each run on dev. They do not create additional fine-tuning techniques for requirement counts.

**Compare:** use the same official dev split and primary metric convention. Inspect `per_label_metrics.csv`,
`rare_label_analysis.json`, `dev_errors.json`, `chunk_coverage.json` and train/dev curves. The best model
per origin is listed in `best_per_architecture.json`; retain architecture-specific submissions even if a
cross-architecture ensemble scores higher. `experiment_summary.csv` compares all compatible tags.

The PDF has conflicting submission counts: its general guidelines say six for Phases 2–7, while §6.5 and
§11 specify four. Six core profiles are supplied to cover the stricter count while meeting all four origins,
at least three adaptation methods, four listed approaches, compulsory LoRMA and KD. Actual accepted
submissions and uniqueness must be checked on Codabench; saved ZIPs alone do not prove acceptance.

The scorer's exact handling of absent IDs/ID gaps is not supplied. `reference_range` includes IDs from
zero through the largest gold ID; `reference_present` averages only IDs occurring in gold; `known`
averages the full known catalog. All use zero F1 for zero-denominator labels. Match any published
official scorer rather than selecting whichever convention makes the local number largest.

Train/dev calibrations are selection scores. Private test scores cannot be verified locally without its
labels. Do not tune on test feedback. More trials or epochs need not improve unseen performance.

## 15. Compare compatible runs and select the best run per Phase 1 origin

In [ ]:
import csv
summary = []
for file in PROJECT_DIR.glob('*/metrics.json'):
    folder = file.parent
    if not (folder / 'run_config.json').is_file():
        continue
    cfg = json.loads((folder / 'run_config.json').read_text())
    if cfg['data_fingerprint'] != DATA_FINGERPRINT or cfg['tokenizer_hash'] != TOKENIZER_HASH:
        continue
    if cfg.get('metric_label_scope') != METRIC_LABEL_SCOPE or cfg.get('label_ids') != LABEL_IDS:
        continue
    metrics = json.loads(file.read_text())
    summary.append({**metrics, 'path': str(folder)})
completed = [row for row in summary if row['training_complete']]
methods = {row['method'] for row in completed}
origins = {row['architecture'] for row in completed}
digests = []
for row in completed:
    path = Path(row['path']) / 'prediction.json'
    if path.is_file():
        predictions = json.loads(path.read_text())
        validate_submission(predictions, DATA['test'])
        digests.append(hashlib.sha256(json.dumps(predictions, sort_keys=True).encode()).hexdigest())
coverage = {'four_completed_runs': len(completed) >= 4, 'six_completed_runs': len(completed) >= 6,
            'six_distinct_prediction_contents': len(set(digests)) >= 6,
            'all_four_phase1_origins': origins >= set(MODEL_CLASSES),
            'three_adaptation_methods_excluding_kd': len(methods & {'full', 'partial', 'lorma', 'adapters', 'gradual'}) >= 3,
            'four_listed_techniques': len(methods & {'full', 'partial', 'lorma', 'adapters', 'kd'}) >= 4,
            'compulsory_lorma': 'lorma' in methods, 'compulsory_kd': 'kd' in methods,
            'codabench_acceptance_verified': False}
save_json(coverage, PROJECT_DIR / 'coverage.json')
best_per_origin = {}
for row in sorted(completed, key=lambda item: item['macro_f1'], reverse=True):
    best_per_origin.setdefault(row['architecture'], {'run_name': row['run_name'], 'macro_f1': row['macro_f1'],
                                'method': row['method'], 'submission': str(Path(row['path']) / 'submission.zip')})
save_json(best_per_origin, PROJECT_DIR / 'best_per_architecture.json')
columns = ['run_name', 'architecture', 'method', 'macro_f1', 'micro_f1', 'subset_accuracy', 'baseline_macro_f1',
           'best_epoch', 'total_parameters', 'training_seconds', 'peak_gpu_memory_bytes', 'training_complete', 'path']
with (PROJECT_DIR / 'experiment_summary.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=columns); writer.writeheader()
    for row in sorted(summary, key=lambda item: item['macro_f1'], reverse=True):
        writer.writerow({key: row.get(key) for key in columns})
        print(row['run_name'], 'DEV Macro-F1=', round(row['macro_f1'], 6), 'Micro-F1=', round(row['micro_f1'], 6),
              'Subset Accuracy=', round(row['subset_accuracy'], 6), 'complete=', row['training_complete'])
print('Best per Phase 1 origin:', best_per_origin)
print('Coverage (local artifacts, not verified competition submissions):', coverage)

References and provenance: local `Capstone Document_edited.pdf` §11, professor's
`CS781_Phase1_Starter_Code.ipynb` for workflow, your Phase 1 compatible definitions and Phase 4's
checkpoint/chunking/resume implementation. This notebook contains all runtime code.

LoRMA+ uses identity-plus-low-rank multiplication of frozen attention projection weights.
The residual adapter has one post-block bottleneck. KD matches independent Bernoulli statute targets.
Keep the course-required LLM declaration and base your analysis on measured runs, not synthetic tests.